# Phish — RF:20partitions, fixed configuration

**FIVE_MODELS v1:20split seeds2001–2020 ×5model seeds12–16 ×4arms =400fits. No hyperparameter search.**

Arms: X14(14),Pure36(36),X14+Pure36(50),X14+SIFE44(58).
Meta-IFD explicitly excludes Pure36-only because its public generator requires original source14.
Use only `dataRaw` and `dataPhish`/`dataHEIG`. No notebook00 or code/model Input mount.

This is the strict unique-address SIFE study, separate from the released-mask reference baseline.
Every arm shares the same partition IDs,model seeds and sampled topology. Configurations are fixed before this run.
All requested fitted arms on all20partitions are saved and locked before TEST is sampled or scored.

Pure36 retains distributional value information; it is not money-information-free. This static transductive
repeated holdout reuses one corpus per task; it is not temporal validation or Phish-to-Ponzi transfer.

Five models only: Meta-IFD,GAT,GraphSAGE,LightGBM,RF. Historical LR stays outside this main study.
LightGBM trains/predicts named DataFrames with exact feature-order validation; no warning suppression.


## Environment — ENVFIX1

Install missing packages only; preserve existing Torch/CUDA. GNNs test actual native heterogeneous sampling and continue after successful installation. Restart is requested only if a fresh interpreter passes but the current kernel still fails.

In [ ]:
import importlib.util
import subprocess
import sys
INSTALL_MISSING_PACKAGES=True
requirements={'numpy':'numpy>=1.24,<3','pandas':'pandas>=2,<4','scipy':'scipy>=1.11,<2',
              'sklearn':'scikit-learn>=1.3,<2','joblib':'joblib>=1.3,<2',
              'threadpoolctl':'threadpoolctl>=3,<4','torch':'torch>=2.1',
              'torch_geometric':'torch-geometric>=2.4,<3'}
for module,requirement in requirements.items():
    if importlib.util.find_spec(module) is None:
        if not INSTALL_MISSING_PACKAGES:raise RuntimeError(f'Install {requirement} first')
        subprocess.check_call([sys.executable,'-m','pip','install',requirement])
import torch
print('Python',sys.version.split()[0],'| Torch',torch.__version__)


## Configuration

Full is the paper-study budget. Smoke uses one partition/one model seed and two GNN epochs with fanout2; it has a separate output and is not a benchmark table. Keep WORK_ROOT for resume.

In [ ]:
import os
import time
from pathlib import Path
DATASET='Phish'
MODEL='rf'
RUN_MODE='full'
INPUT_ROOT=Path('/kaggle/input') if Path('/kaggle/input').is_dir() else Path.cwd()
DATA_RAW=Path(os.environ.get('SIFE20_RAW_INPUT',str(INPUT_ROOT)))
DATA_PHISH=Path(os.environ.get('SIFE20_PROCESSED_INPUT',str(INPUT_ROOT)))
OUTPUT_ROOT=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
WORK_ROOT=Path(os.environ.get('SIFE20_WORK_ROOT',str(OUTPUT_ROOT/'sife20_native_five_fixed_v1')))/DATASET/MODEL
DATA_OUTPUT=WORK_ROOT/'data'
JOB_DIR=WORK_ROOT/'results'/RUN_MODE
DEVICE='cpu' if MODEL in ('lr','rf','lightgbm') else ('cuda:0' if torch.cuda.is_available() else 'cpu')
CPU_THREADS=2
MAX_SESSION_HOURS=None  # Optional soft budget between fits. Keep WORK_ROOT to resume.
INCLUDE_LARGE_CACHES_IN_EXPORT=False  # True makes a larger snapshot for exact cached resume.
SESSION_DEADLINE=None if MAX_SESSION_HOURS is None else time.monotonic()+3600*MAX_SESSION_HOURS
print('Dataset/model:',DATASET,MODEL,'| mode:',RUN_MODE,'| device:',DEVICE)
print('Output:',WORK_ROOT)


## Implementation: utilities

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.imports
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import torch


### require

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.require
def require(condition, message):
    if not condition:
        raise ValueError(message)


### sha256

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.sha256
def sha256(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()


### json_hash

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.json_hash
def json_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':'), allow_nan=False)
    return hashlib.sha256(payload.encode('utf-8')).hexdigest()


### write_json

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.write_json
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.partial')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


### read_json

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.read_json
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


### seed_everything

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.seed_everything
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### runtime_information

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.runtime_information
def runtime_information(device):
    packages = ['numpy', 'pandas', 'scipy', 'scikit-learn', 'torch', 'torch-geometric','joblib','threadpoolctl']
    versions = {name: importlib.metadata.version(name) for name in packages}
    try:
        versions['lightgbm'] = importlib.metadata.version('lightgbm')
    except importlib.metadata.PackageNotFoundError:
        versions['lightgbm'] = None
    for name in ['pyg-lib','torch-sparse','torch-scatter']:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return dict(python=platform.python_version(), platform=platform.platform(),
                packages=versions,
                device=str(device), cuda=torch.version.cuda,
                cpu_threads=torch.get_num_threads(),
                gpu=torch.cuda.get_device_name(device) if str(device).startswith('cuda') else None,
                sampler='pyg_neighbor_loader_directional_v1',
                bitwise_cross_platform_equivalence_claimed=False)


### seal_files

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.seal_files
def seal_files(root, files):
    root = Path(root)
    return {Path(p).relative_to(root).as_posix(): sha256(p) for p in files}


### verify_files

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.verify_files
def verify_files(root, manifest):
    for relative, expected in manifest.items():
        require(sha256(Path(root)/relative) == expected, f'Frozen file changed: {relative}')


### atomic_torch_save

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: utilities.atomic_torch_save
def atomic_torch_save(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.partial')
    torch.save(value, temporary)
    os.replace(temporary, path)


## Implementation: features

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: features.imports
import numpy as np
FEATURE_VERSION = 'sife44_distribution_v9_formulas_v1'
CHANNELS = (('in','call'), ('out','call'), ('in','trans'), ('out','trans'))
DISTRIBUTION_STATS = ('entropy','entropy_norm','hhi','jsd_freq_val','max_share',
                      'std_log_val','mean_tx_val')
SIFE44_NAMES = tuple(
    [f'sife_{rel}_{direction}_{stat}' for direction,rel in CHANNELS for stat in DISTRIBUTION_STATS]
    + [f'sife_{rel}_{stat}' for rel in ('call','trans')
       for stat in ('funnel_entropy','funnel_hhi','asym_deg','asym_flow')]
    + [f'sife_{rel}_{direction}_{stat}' for direction,rel in CHANNELS
       for stat in ('ca_edge_ratio','ca_count_ratio')])
PURE36_INDICES = tuple(ch*7+j for ch in range(4) for j in range(5))+tuple(range(28,44))
INTENSITY8_INDICES = tuple(ch*7+j for ch in range(4) for j in (5,6))
ARMS = ('X14','Pure36','X14_Pure36','X14_SIFE44')


### safe_ratio

Return zero for a zero denominator, retaining signed contrast numerators.

In [ ]:
# SIFE20_SOURCE_CELL: features.safe_ratio
def safe_ratio(numerator, denominator):
    """Return zero for a zero denominator, retaining signed contrast numerators."""
    return np.divide(numerator,denominator,out=np.zeros_like(numerator,dtype=np.float64),
                     where=denominator>0)


### summarize_channel

Two passes over (local_id,count,sum,partner_is_CA) for a direction/relation.

For a nonempty zero-mass channel, use a uniform distribution on its records.
For an absent channel, all returned statistics are zero. JSD uses natural logs.

In [ ]:
# SIFE20_SOURCE_CELL: features.summarize_channel
def summarize_channel(n, record_factory):
    """Two passes over (local_id,count,sum,partner_is_CA) for a direction/relation.

    For a nonempty zero-mass channel, use a uniform distribution on its records.
    For an absent channel, all returned statistics are zero. JSD uses natural logs.
    """
    rows,counts,sums,maximum,logs,ca_rows,ca_counts=np.zeros((7,n),dtype=np.float64)
    for idx,c_raw,s_raw,partner_ca in record_factory():
        c=np.asarray(c_raw,dtype=np.float64)
        s=np.asarray(s_raw,dtype=np.float64)
        if not (np.isfinite(c).all() and np.isfinite(s).all() and (c>=0).all() and (s>=0).all()):
            raise ValueError('Incident count/sum weights must be finite and nonnegative')
        rows+=np.bincount(idx,minlength=n)
        counts+=np.bincount(idx,weights=c,minlength=n)
        sums+=np.bincount(idx,weights=s,minlength=n)
        logs+=np.bincount(idx,weights=np.log1p(s),minlength=n)
        np.maximum.at(maximum,idx,s)
        if partner_ca:
            ca_rows+=np.bincount(idx,minlength=n)
            ca_counts+=np.bincount(idx,weights=c,minlength=n)
    entropy,hhi,jsd,variance=np.zeros((4,n),dtype=np.float64)
    mean_log=safe_ratio(logs,rows)
    for idx,c_raw,s_raw,_ in record_factory():
        c=np.asarray(c_raw,dtype=np.float64)
        s=np.asarray(s_raw,dtype=np.float64)
        pc=safe_ratio(c,counts[idx])
        ps=safe_ratio(s,sums[idx])
        pc=np.where(counts[idx]>0,pc,1/rows[idx])
        ps=np.where(sums[idx]>0,ps,1/rows[idx])
        midpoint=(pc+ps)/2
        terms=np.zeros_like(ps)
        positive=ps>0
        terms[positive]=-ps[positive]*np.log(ps[positive])
        entropy+=np.bincount(idx,weights=terms,minlength=n)
        hhi+=np.bincount(idx,weights=ps**2,minlength=n)
        terms=np.zeros_like(ps)
        for probability in (pc,ps):
            positive=probability>0
            terms[positive]+=.5*probability[positive]*np.log(probability[positive]/midpoint[positive])
        jsd+=np.bincount(idx,weights=terms,minlength=n)
        variance+=np.bincount(idx,weights=(np.log1p(s)-mean_log[idx])**2,minlength=n)
    normalized=safe_ratio(entropy,np.log(np.maximum(rows,2)))*(rows>1)
    dominance=np.where(sums>0,safe_ratio(maximum,sums),safe_ratio(np.ones(n),rows))
    distribution=np.column_stack((entropy,normalized,hhi,np.clip(jsd,0,np.log(2)),dominance,
                    np.sqrt(safe_ratio(variance,rows)),np.log1p(safe_ratio(sums,counts))))
    metadata=dict(rows=rows,counts=counts,sums=sums,entropy=entropy,hhi=hhi,
                  ca_rows=safe_ratio(ca_rows,rows),ca_counts=safe_ratio(ca_counts,counts))
    return distribution,metadata


### extract_sife44

Compute all incident records for sorted IDs, independent of labels/source14.

edges maps (source_type,call|trans,destination_type) to arrays edge_index[2,E]
and edge_attr[E,2] in the same order; the attribute columns are count and sum.
Already aggregated typed counterpart pairs must retain their original identity.
The prepared-data stage separately verifies pair uniqueness and raw alignment.

In [ ]:
# SIFE20_SOURCE_CELL: features.extract_sife44
def extract_sife44(node_counts, edges, node_type, ids, chunk_size=250000):
    """Compute all incident records for sorted IDs, independent of labels/source14.

    edges maps (source_type,call|trans,destination_type) to arrays edge_index[2,E]
    and edge_attr[E,2] in the same order; the attribute columns are count and sum.
    Already aggregated typed counterpart pairs must retain their original identity.
    The prepared-data stage separately verifies pair uniqueness and raw alignment.
    """
    if node_type not in ('EOA','CA') or node_type not in node_counts:
        raise ValueError('Known EOA/CA target node type required')
    ids=np.asarray(ids)
    if ids.ndim!=1 or not np.issubdtype(ids.dtype,np.integer) or not len(ids):
        raise ValueError('Nonempty integer node ID vector required')
    ids=ids.astype(np.int64,copy=False)
    if np.any(np.diff(ids)<=0) or ids[0]<0 or ids[-1]>=node_counts[node_type]:
        raise ValueError('Node IDs must be sorted, unique and in range')
    if not isinstance(chunk_size,int) or chunk_size<1:
        raise ValueError('Positive integer chunk size required')
    for (src,rel,dst),record in edges.items():
        ix=record['edge_index']
        attr=record.get('edge_attr')
        if src not in node_counts or dst not in node_counts or rel not in ('call','trans'):
            raise ValueError('Unsupported typed relation')
        if ix.ndim!=2 or ix.shape[0]!=2 or not np.issubdtype(ix.dtype,np.integer):
            raise ValueError('edge_index must have integer shape[2,E]')
        if attr is None or attr.shape!=(ix.shape[1],2):
            raise ValueError('Aligned count/sum edge_attr[E,2] is required')
    lookup=np.full(node_counts[node_type],-1,dtype=np.int32)
    lookup[ids]=np.arange(len(ids),dtype=np.int32)
    values=np.zeros((len(ids),44),dtype=np.float64)
    metadata={}
    for channel,(direction,relation) in enumerate(CHANNELS):
        def factory():
            for (src,rel,dst),record in edges.items():
                if rel!=relation or (src if direction=='out' else dst)!=node_type:
                    continue
                axis=0 if direction=='out' else 1
                partner=dst if direction=='out' else src
                for start in range(0,record['edge_index'].shape[1],chunk_size):
                    endpoints=record['edge_index'][axis,start:start+chunk_size]
                    if endpoints.size and (endpoints.min()<0 or endpoints.max()>=len(lookup)):
                        raise ValueError('Target endpoint out of range')
                    local=lookup[endpoints]
                    selected=local>=0
                    attr=record['edge_attr'][start:start+chunk_size][selected]
                    yield local[selected],attr[:,0],attr[:,1],partner=='CA'
        dist,meta=summarize_channel(len(ids),factory)
        values[:,channel*7:(channel+1)*7]=dist
        metadata[(direction,relation)]=meta
        values[:,36+channel*2]=meta['ca_rows']
        values[:,37+channel*2]=meta['ca_counts']
    for j,relation in enumerate(('call','trans')):
        incoming=metadata[('in',relation)]
        outgoing=metadata[('out',relation)]
        values[:,28+j*4:32+j*4]=np.column_stack((
            incoming['entropy']-outgoing['entropy'],outgoing['hhi']-incoming['hhi'],
            safe_ratio(incoming['rows']-outgoing['rows'],incoming['rows']+outgoing['rows']),
            safe_ratio(incoming['sums']-outgoing['sums'],incoming['sums']+outgoing['sums'])))
    activity=np.column_stack((np.log1p(sum(m['counts'] for m in metadata.values())),
                              np.log1p(sum(m['sums'] for m in metadata.values()))))
    if not np.isfinite(values).all() or not np.isfinite(activity).all():
        raise ValueError('Nonfinite feature output')
    return dict(ids=ids.copy(),sife44=values,activity=activity)


### select_inputs

Select explicitly named blocks; no historical cache column offsets.

In [ ]:
# SIFE20_SOURCE_CELL: features.select_inputs
def select_inputs(source14, sife44, arm, model=None):
    """Select explicitly named blocks; no historical cache column offsets."""
    if arm not in ARMS:
        raise ValueError('Unknown feature arm')
    if arm=='Pure36' and model=='metaifd':
        raise ValueError('Meta-IFD generator requires source14; Pure36-only is unsupported')
    sife44=np.asarray(sife44)
    if sife44.ndim!=2 or sife44.shape[1]!=44 or not np.isfinite(sife44).all():
        raise ValueError('Finite SIFE44 matrix required')
    if arm=='Pure36':
        return sife44[:,PURE36_INDICES].copy()
    source14=np.asarray(source14)
    if source14.shape!=(len(sife44),14) or not np.isfinite(source14).all():
        raise ValueError('Aligned finite source14 required')
    if arm=='X14':
        return source14.copy()
    extra=sife44[:,PURE36_INDICES] if arm=='X14_Pure36' else sife44
    return np.column_stack((source14,extra))


## Implementation: data

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: data.imports
from contextlib import contextmanager
import gc
import hashlib
import io
import os
from pathlib import Path
import shutil
import zipfile
import numpy as np
import pandas as pd
import torch


### HashedStream

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: data.HashedStream
class HashedStream(io.RawIOBase):
    def __init__(self,stream):
        self.stream=stream;self.digest=hashlib.sha256()
    def readable(self):return True
    def readinto(self,buffer):
        data=self.stream.read(len(buffer));buffer[:len(data)]=data
        self.digest.update(data);return len(data)
    def close(self):
        self.stream.close();super().close()


### resolve_input

Accept a ZIP or extracted directory, including nested Kaggle dataset mounts.

In [ ]:
# SIFE20_SOURCE_CELL: data.resolve_input
def resolve_input(root,dataset,kind):
    """Accept a ZIP or extracted directory, including nested Kaggle dataset mounts."""
    root=Path(root).expanduser().resolve()
    if not root.exists():raise ValueError(f'Missing {kind} input: {root}')
    if root.is_file():return root
    if kind=='raw':
        archives=sorted(root.rglob('OneDrive_2026-09-18.zip'))
        if len(archives)==1:return archives[0]
        marker='EOA_labeled.csv' if dataset=='Phish' else 'CA_labeled.csv'
        folders=[p.parent.parent for p in root.rglob(marker) if p.parent.name==dataset+'_unbalance']
        if len(folders)==1:return folders[0]
    else:
        matches=[p for p in root.rglob('data.pt') if dataset.lower() in [s.lower() for s in p.parts[:-1]]]
        if len(matches)==1:return matches[0]
        archives=sorted(root.rglob('HEIG.zip'))
        if len(archives)==1:return archives[0]
    raise ValueError(f'Need one {dataset} {kind} source; set its exact path in configuration')


### open_raw_member

Open a canonical member from ZIP or the extracted Meta-IFD data folder.

In [ ]:
# SIFE20_SOURCE_CELL: data.open_raw_member
@contextmanager
def open_raw_member(source,member):
    """Open a canonical member from ZIP or the extracted Meta-IFD data folder."""
    source=Path(source)
    if source.is_file():
        with zipfile.ZipFile(source) as archive:
            with archive.open(member) as stream:yield stream
    else:
        prefix='Meta-IFD data/'
        if not member.startswith(prefix):raise ValueError('Invalid raw member')
        path=(source/member[len(prefix):]).resolve()
        if not path.is_relative_to(source.resolve()):raise ValueError('Unsafe raw member')
        with path.open('rb') as stream:yield stream


### obtain_graph

Verify known graph bytes before PyG deserialization; use direct files read-only.

In [ ]:
# SIFE20_SOURCE_CELL: data.obtain_graph
def obtain_graph(source,output,dataset,expected_hash):
    """Verify known graph bytes before PyG deserialization; use direct files read-only."""
    source=Path(source)
    if source.suffix.lower()=='.pt':path=source
    else:
        path=Path(output)/'graph/data.pt'
        if not path.exists():
            path.parent.mkdir(parents=True,exist_ok=True)
            temporary=path.with_suffix('.pt.partial')
            with zipfile.ZipFile(source) as archive:
                candidates=[n for n in archive.namelist() if n.endswith(f'{dataset}/processed/data.pt')]
                if len(candidates)!=1:raise ValueError('Need one target graph inside HEIG.zip')
                with archive.open(candidates[0]) as stream,temporary.open('wb') as target:
                    shutil.copyfileobj(stream,target,8*1024*1024)
            if sha256(temporary)!=expected_hash:raise ValueError('Wrong extracted graph')
            os.replace(temporary,path)
    if sha256(path)!=expected_hash:raise ValueError('Processed graph is not the locked dataset release')
    return path


### verified_raw_cohort

Verify the raw labelled cohort table against its frozen byte hash.

In [ ]:
# SIFE20_SOURCE_CELL: data.verified_raw_cohort
def verified_raw_cohort(source,dataset,reference):
    """Verify the raw labelled cohort table against its frozen byte hash."""
    name='EOA_labeled.csv' if dataset=='Phish' else 'CA_labeled.csv'
    member=f'Meta-IFD data/{dataset}_unbalance/{name}'
    expected=reference['raw_members'][member]
    with open_raw_member(source,member) as raw:
        content=raw.read()
    if len(content)!=expected['bytes'] or hashlib.sha256(content).hexdigest()!=expected['sha256']:
        raise ValueError('Raw labelled table changed')
    frame=pd.read_csv(io.BytesIO(content))
    column='name' if dataset=='Phish' else 'Address'
    addresses=frame[column].astype(str).str.lower().to_numpy()
    mapped=np.asarray(reference['addresses'])
    if not set(mapped)<=set(addresses):raise ValueError('Frozen address mapping is absent from raw labelled table')
    if dataset=='Phish':
        positive=set(addresses[:1207]);negative=set(addresses[1207:])
        for address,label in zip(mapped,reference['labels']):
            if address not in (positive if label else negative):raise ValueError('Raw/source cohort label-group mismatch')
    return expected['sha256']


### restore_float64_attributes

Restore all six raw Ponzi arrays, checking raw bytes/order and exact NPY hashes.

In [ ]:
# SIFE20_SOURCE_CELL: data.restore_float64_attributes
def restore_float64_attributes(raw_source,output,reference):
    """Restore all six raw Ponzi arrays, checking raw bytes/order and exact NPY hashes."""
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    report={}
    for key,entry in reference['attribute_provenance']['relations'].items():
        target=output/entry['file']
        if target.exists():
            if sha256(target)!=entry['sha256']:raise ValueError('Cached Ponzi sidecar changed')
        else:
            src,rel,dst=key.split('__')
            member=f'Meta-IFD data/Ponzi_unbalance/edge/{rel}_{src.lower()}_{dst.lower()}.csv'
            expected=reference['raw_members'][member]
            temporary=target.with_suffix('.npy.partial')
            values=np.lib.format.open_memmap(temporary,mode='w+',dtype=np.float64,shape=(entry['rows'],2))
            cursor=0
            print('Restore raw float64:',key,flush=True)
            with open_raw_member(raw_source,member) as raw:
                reader=HashedStream(raw)
                with io.BufferedReader(reader,buffer_size=8*1024*1024) as stream:
                    for frame in pd.read_csv(stream,usecols=['count','sum'],chunksize=300000):
                        stop=cursor+len(frame)
                        if stop>len(values):raise ValueError('Too many raw rows')
                        chunk=frame[['count','sum']].to_numpy(np.float64)
                        if not np.isfinite(chunk).all() or (chunk<0).any():raise ValueError('Invalid raw edge weights')
                        values[cursor:stop]=chunk;cursor=stop
            if cursor!=len(values) or reader.digest.hexdigest()!=expected['sha256']:raise ValueError('Raw CSV checksum/row mismatch')
            values.flush();values._mmap.close();del values
            if sha256(temporary)!=entry['sha256']:raise ValueError('Restored array differs from raw-to-graph audit')
            os.replace(temporary,target)
        report[key]=entry['sha256']
    return report


### prepare_data

Verify two data inputs, restore Ponzi weights, compute roots and remove duplicate addresses.

In [ ]:
# SIFE20_SOURCE_CELL: data.prepare_data
def prepare_data(raw_source,processed_source,output,dataset,reference,source_signature):
    """Verify two data inputs, restore Ponzi weights, compute roots and remove duplicate addresses."""
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    path=obtain_graph(processed_source,output,dataset,reference['graph_sha256'])
    raw_hash=verified_raw_cohort(raw_source,dataset,reference)
    payload=torch.load(str(path),map_location='cpu',mmap=True,weights_only=False)
    graph=payload[0] if isinstance(payload,tuple) else payload
    require(set(graph.node_types)=={'EOA','CA'},'Expected released EOA/CA graph')
    require(all(graph[nt].x.shape[1]==14 for nt in graph.node_types),'Expected14source columns')
    target='EOA' if dataset=='Phish' else 'CA'
    ids=np.asarray(reference['ids'],dtype=np.int64);labels=np.asarray(reference['labels'],dtype=np.int64)
    addresses=np.asarray(reference['addresses'])
    require(np.array_equal(graph[target].y[torch.from_numpy(ids)].numpy(),labels),'Graph labels changed')
    sidecars=restore_float64_attributes(raw_source,output/'float64_attributes',reference) if dataset=='Ponzi' else {}
    edges={}
    for et in graph.edge_types:
        if dataset=='Ponzi':
            entry=reference['attribute_provenance']['relations']['__'.join(et)]
            attributes=np.load(output/'float64_attributes'/entry['file'],mmap_mode='r',allow_pickle=False)
        else:attributes=graph[et].edge_attr.numpy()
        edges[et]=dict(edge_index=graph[et].edge_index.numpy(),edge_attr=attributes)
    inputs=dict(dataset=dataset,graph_sha256=reference['graph_sha256'],raw_label_sha256=raw_hash,
        reference_hash=json_hash(reference),source_signature=source_signature,feature_version=FEATURE_VERSION,
        sidecar_hashes=sidecars)
    feature_path=output/'roots.npz';manifest_path=output/'data_manifest.json'
    if manifest_path.exists():
        manifest=read_json(manifest_path)
        require(manifest['inputs']==inputs and sha256(feature_path)==manifest['roots_sha256'],
                'Data/code cache changed; use a new work folder')
        with np.load(feature_path,allow_pickle=False) as stored:roots={key:stored[key].copy() for key in stored.files}
    else:
        _,inverse,counts=np.unique(addresses,return_inverse=True,return_counts=True)
        keep=counts[inverse]==1
        require(int(keep.sum())=={'Phish':2761,'Ponzi':1342}[dataset],'Unexpected strict cohort size')
        kept_ids=ids[keep]
        print('Compute exact root SIFE44:',dataset,len(kept_ids),'roots',flush=True)
        result=extract_sife44({nt:graph[nt].num_nodes for nt in graph.node_types},edges,target,kept_ids)
        roots=dict(ids=kept_ids,labels=labels[keep],addresses=addresses[keep],
                   source14=graph[target].x[torch.from_numpy(kept_ids)].numpy(),
                   sife44=result['sife44'],activity=result['activity'])
        np.savez_compressed(feature_path,**roots)
        manifest=dict(inputs=inputs,roots_sha256=sha256(feature_path),cohort_size=len(kept_ids),
            positives=int(labels[keep].sum()),excluded_ids=ids[~keep].tolist(),
            cohort_policy='exclude every supervised row belonging to any duplicated labelled address; graph topology unchanged',
            mapping_scope='audited labelled-root mapping bound to graph/raw byte hashes; existing raw-to-graph alignment audit',
            setting='static transductive; no verified timestamps or prediction cutoff',
            feature_scope='full incident typed edge records; same numeric definition for all models')
        write_json(manifest_path,manifest)
    require(len(set(roots['addresses']))==len(roots['ids']),'Strict root cohort contains duplicate addresses')
    return dict(graph=graph,edges=edges,roots=roots,manifest=manifest,
                node_counts={nt:graph[nt].num_nodes for nt in graph.node_types},data_output=output)


## Implementation: sampling

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: sampling.imports
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch_geometric.data import HeteroData
from torch_geometric.loader import NeighborLoader
import torch_geometric.typing


### save_pyg_role_batches

Cache actual PyG NeighborLoader samples; root IDs precede neighbors, as in author code.

In [ ]:
# SIFE20_SOURCE_CELL: sampling.save_pyg_role_batches
def save_pyg_role_batches(graph, membership, target_type, role, output_dir,
                          sampler_seed, fanouts=(100,100), batch_size=512):
    """Cache actual PyG NeighborLoader samples; root IDs precede neighbors, as in author code."""
    require(torch_geometric.typing.WITH_PYG_LIB or torch_geometric.typing.WITH_TORCH_SPARSE,
            'NeighborLoader needs a working pyg-lib or torch-sparse installation for this Torch build; no sampler fallback is allowed')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    # Sampling needs topology and node counts, not full features or edge attributes.
    topology = HeteroData()
    for nt in graph.node_types:
        topology[nt].num_nodes = graph[nt].num_nodes
    for et in graph.edge_types:
        topology[et].edge_index = graph[et].edge_index
    roots = torch.tensor(membership.node_id.to_numpy(copy=True), dtype=torch.long)
    records = []
    # Explicit per-role seeds are a reproducibility correction, not recovered author RNG.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(sampler_seed)
        loader = NeighborLoader(topology, num_neighbors=list(fanouts),
                    input_nodes=(target_type, roots), batch_size=batch_size,
                    shuffle=role != 'test', replace=False, subgraph_type='directional',
                    num_workers=0, filter_per_worker=False)
        for sampled in loader:
            size = int(sampled[target_type].batch_size)
            ids = {nt: sampled[nt].n_id.cpu().clone() for nt in graph.node_types}
            seed_ids = ids[target_type][:size]
            batch = dict(node_ids=ids,
                         edge_index={et: sampled[et].edge_index.cpu().clone() for et in graph.edge_types},
                         seed_ids=seed_ids, seed_labels=graph[target_type].y[seed_ids].cpu().clone(),
                         target_type=target_type)
            require(torch.equal(roots[sampled[target_type].input_id], seed_ids), 'PyG input/root identity mismatch')
            path = output_dir/f'batch_{len(records):04d}.pt'
            atomic_torch_save(batch, path)
            records.append(dict(file=path.name, sha256=sha256(path), seeds=seed_ids.tolist()))
    actual = [node for record in records for node in record['seeds']]
    require(len(actual) == len(set(actual)) == len(membership) and set(actual) == set(membership.node_id),
            'PyG sampled root coverage mismatch')
    write_json(output_dir/'index.json', dict(role=role, seed=sampler_seed,
        backend='pyg_neighbor_loader_directional_v1', fanouts=list(fanouts),
        batch_size=batch_size, records=records))
    return load_role_batches(output_dir)


### load_role_batches

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: sampling.load_role_batches
def load_role_batches(output_dir):
    output_dir = Path(output_dir)
    index = read_json(output_dir/'index.json')
    records = []
    for row in index['records']:
        path = output_dir/row['file']
        require(sha256(path) == row['sha256'], 'Sampled topology cache changed')
        records.append(dict(row, path=str(path)))
    return records


## Implementation: attention_conv

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: attention_conv.imports
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import Linear
from torch_geometric.nn.inits import reset


### my_conv

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: attention_conv.my_conv
class my_conv(nn.Module):
    def __init__(self, in_channels, out_channels, metadata, group='sum'):
        super().__init__()
        if not isinstance(in_channels, dict):
            in_channels = {nt: in_channels for nt in metadata[0]}
        if group != 'sum':
            raise ValueError('The released model uses relation sum aggregation')
        self.out_channels = out_channels
        self.v_lin = nn.ModuleDict({nt: Linear(width, out_channels) for nt,width in in_channels.items()})
        self.a_lin = nn.ModuleDict({nt: Linear(width+out_channels, out_channels) for nt,width in in_channels.items()})
        self.skip = nn.ParameterDict({nt: nn.Parameter(torch.ones(1)) for nt in in_channels})

    def reset_parameters(self):
        reset(self.v_lin)
        reset(self.a_lin)
        for parameter in self.skip.values():
            nn.init.ones_(parameter)

    def forward(self, x_dict, edge_index_dict):
        projected = {nt: self.v_lin[nt](values) for nt,values in x_dict.items()}
        incoming = {nt: [] for nt in x_dict}
        for (src, _, dst), edges in edge_index_dict.items():
            summed = projected[dst].new_zeros((len(projected[dst]), self.out_channels))
            summed.index_add_(0, edges[1], projected[src][edges[0]])
            incoming[dst].append(summed)
        output = {}
        for nt, values in x_dict.items():
            total = torch.stack(incoming[nt], dim=0).sum(dim=0)
            output[nt] = self.a_lin[nt](torch.tanh(torch.cat([total, values], dim=1)))
        return output


## Implementation: models_meta

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: models_meta.imports
from typing import Optional, List
import torch
import torch.nn as nn
from torch import Tensor
from torch_geometric.nn import Linear
import torch.nn.functional as F
from torch_geometric.nn.inits import glorot, reset
from torch_geometric.nn import MessagePassing


### group

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: models_meta.group
def group(xs: List[Tensor], aggr: Optional[str]) -> Optional[Tensor]:
    if len(xs) == 0:
        return None
    elif aggr is None:
        return torch.stack(xs, dim=1)
    elif len(xs) == 1:
        return xs[0]
    elif aggr == "cat":
        return torch.cat(xs, dim=-1)
    else:
        out = torch.stack(xs, dim=0)
        out = getattr(torch, aggr)(out, dim=0)
        out = out[0] if isinstance(out, tuple) else out
        return out


### TripletLoss

Compute normal triplet loss or soft margin triplet loss given triplets

In [ ]:
# SIFE20_SOURCE_CELL: models_meta.TripletLoss
class TripletLoss(nn.Module):
    '''
    Compute normal triplet loss or soft margin triplet loss given triplets
    '''

    def __init__(self, margin=None):
        super(TripletLoss, self).__init__()
        self.margin = margin
        if self.margin is None:  # if no margin assigned, use soft-margin
            self.Loss = nn.SoftMarginLoss()
        else:
            self.Loss = nn.TripletMarginLoss(margin=margin, p=2)

    def forward(self, anchor, pos, neg):
        if self.margin is None:
            num_samples = anchor.shape[0]
            y = torch.ones((num_samples, 1)).view(-1)
            if anchor.is_cuda: y = y.cuda()
            ap_dist = torch.norm(anchor - pos, 2, dim=1).view(-1)
            an_dist = torch.norm(anchor - neg, 2, dim=1).view(-1)
            loss = self.Loss(an_dist - ap_dist, y)
        else:
            loss = self.Loss(anchor, pos, neg)

        return loss


### HMSL

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: models_meta.HMSL
class HMSL(nn.Module):
    def __init__(self, hidden, out_channels, data, concat, target_node, input_width=14):
        super().__init__()
        self.concat_num = concat
        self.target = target_node
        self.loss_co = TripletLoss(margin=0.3)

        self.lin_dict = torch.nn.ModuleDict()
        self.lin_dict_mean = torch.nn.ModuleDict()
        for node_type in data.node_types:
            self.lin_dict[node_type] = Linear(input_width, hidden)
            self.lin_dict_mean[node_type] = Linear(hidden, hidden)
        self.lin_out = Linear(hidden, out_channels)

        self.k_lin = torch.nn.ModuleDict()
        self.q_lin = torch.nn.ModuleDict()
        self.v_lin = torch.nn.ModuleDict()

        for node_type in data.node_types:
            self.k_lin[node_type] = Linear(hidden, hidden)
            self.q_lin[node_type] = Linear(hidden, hidden)
            self.v_lin[node_type] = Linear(hidden, hidden)

        self.conv = my_conv(hidden, hidden, metadata=data.metadata())
        self.conv1 = my_conv(hidden, hidden, metadata=data.metadata())

    def reset_parameters(self):
        reset(self.lin_dict)
        reset(self.lin_dict_mean)
        reset(self.k_lin)
        reset(self.q_lin)
        reset(self.v_lin)
        reset(self.conv)
        reset(self.conv1)
        self.lin_out.reset_parameters()

    def forward(self, x_dict, edge_index, sife_dict=None):
        CA_hidden_ls = []
        EOA_hidden_ls = []

        ca_out_ls = []
        eoa_out_ls = []
        out_dict = {}
        for k in range(self.concat_num):
            x_dict_ = {}
            for node_type, x in x_dict.items():
                layer = self.lin_dict[node_type]
                if sife_dict is None:
                    encoded = layer(x[k])
                else:
                    # Algebraically the same as concatenating SIFE44 to every
                    # generated/original interaction view; avoid 16 large copies.
                    encoded = F.linear(x[k], layer.weight[:, :14], layer.bias)
                    extra = F.linear(sife_dict[node_type], layer.weight[:, 14:])
                    encoded = encoded + extra.unsqueeze(0)
                x_dict_[node_type] = torch.tanh(encoded)
            CA_hidden_ls.append(x_dict_['CA'])
            EOA_hidden_ls.append(x_dict_['EOA'])
            CA_h = torch.mean(CA_hidden_ls[k], dim=0)
            CA_h = F.tanh(self.lin_dict_mean['CA'](CA_h))
            EOA_h = torch.mean(EOA_hidden_ls[k], dim=0)
            EOA_h = F.tanh(self.lin_dict_mean['EOA'](EOA_h))
            ca_out_ls.append(CA_h)
            eoa_out_ls.append(EOA_h)

        out_dict['CA'] = ca_out_ls
        out_dict['EOA'] = eoa_out_ls

        for node_type, x in out_dict.items():
            out_dict[node_type] = self.attention(x, node_type)

        out_dict = self.conv(out_dict, edge_index)
        out_dict = self.conv1(out_dict, edge_index)

        out = self.lin_out(out_dict[self.target])
        loss = self.contrast_module(CA_hidden_ls, EOA_hidden_ls)

        return out, loss

    def contrast_module(self, CA_hidden_ls, EOA_hidden_ls):
        anchors = []
        pos = []
        neg = []
        for i, z in enumerate(CA_hidden_ls):
            anchors.append(z[1])
            pos.append(z[2])
            neg.append(z[0])

        out_anchors = self.attention(anchors, 'CA')
        out_pos = self.attention(pos, 'CA')
        out_neg = self.attention(neg, 'CA')
        loss_ca = self.loss_co(out_anchors, out_pos, out_neg)

        anchors = []
        pos = []
        neg = []
        for i, z in enumerate(EOA_hidden_ls):
            anchors.append(z[1])
            pos.append(z[2])
            neg.append(z[0])

        out_anchors = self.attention(anchors, 'EOA')
        out_pos = self.attention(pos, 'EOA')
        out_neg = self.attention(neg, 'EOA')
        loss_eoa = self.loss_co(out_anchors, out_pos, out_neg)
        loss = loss_ca + loss_eoa
        return loss

    def attention(self, input, node_type):
        input = torch.stack(input, dim=0).permute(1, 0, 2)
        q = self.q_lin[node_type](input)
        k = self.k_lin[node_type](input)
        v = self.v_lin[node_type](input)
        att = torch.matmul(k, q.transpose(1, 2))
        att = torch.nn.functional.softmax(att, dim=-1)
        v = att @ v
        v = torch.mean(v, dim=1)
        return v


## Implementation: views

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: views.imports
from pathlib import Path
import hashlib
import numpy as np
import torch
from torch import nn


### FrozenInteractionDecoder

Public pretrained conditional decoder: latent50 + source14 + relation6 ->14.

In [ ]:
# SIFE20_SOURCE_CELL: views.FrozenInteractionDecoder
class FrozenInteractionDecoder(nn.Module):
    """Public pretrained conditional decoder: latent50 + source14 + relation6 ->14."""
    def __init__(self):
        super().__init__();self.latent_size=50
        self.MLP=nn.Sequential()
        self.MLP.add_module('L0',nn.Linear(70,32));self.MLP.add_module('A0',nn.Tanh())
        self.MLP.add_module('L_out0',nn.Linear(32,14))
    def forward(self,z,source,relation):
        return torch.sigmoid(self.MLP(torch.cat([z,source,relation],dim=-1)))


### decoder_tensor_fingerprint

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: views.decoder_tensor_fingerprint
def decoder_tensor_fingerprint(state):
    digest=hashlib.sha256()
    for name in sorted(state):
        tensor=state[name].detach().cpu().contiguous()
        digest.update(name.encode());digest.update(str(tuple(tensor.shape)).encode());digest.update(tensor.numpy().tobytes())
    return digest.hexdigest()


### materialize_decoder

Restore visible numeric weights and verify their independent tensor fingerprint.

In [ ]:
# SIFE20_SOURCE_CELL: views.materialize_decoder
def materialize_decoder(output,reference):
    """Restore visible numeric weights and verify their independent tensor fingerprint."""
    state={name:torch.tensor(values,dtype=torch.float32) for name,values in reference['tensors'].items()}
    require(decoder_tensor_fingerprint(state)==reference['tensor_fingerprint'],'Inline public decoder tensors changed')
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    path=output/'public_decoder.pt'
    if path.exists():
        saved=torch.load(path,map_location='cpu',weights_only=True)
        require(decoder_tensor_fingerprint(saved)==reference['tensor_fingerprint'],'Cached decoder changed')
    else:atomic_torch_save(state,path)
    decoder=FrozenInteractionDecoder();decoder.load_state_dict(state,strict=True)
    decoder.eval();decoder.requires_grad_(False)
    return decoder


### author_row_normalize

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: views.author_row_normalize
def author_row_normalize(values):
    inverse=1/values.sum(dim=1);inverse[torch.isinf(inverse)]=0
    return values*inverse[:,None]


### save_meta_views

Cache public-code conditional interaction views; RNG/traversal are declared corrections.

In [ ]:
# SIFE20_SOURCE_CELL: views.save_meta_views
def save_meta_views(graph,records,output,decoder,role_seed,views):
    """Cache public-code conditional interaction views; RNG/traversal are declared corrections."""
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    generator=torch.Generator(device='cpu').manual_seed(role_seed+50000)
    files={}
    for record in records:
        batch=torch.load(record['path'],map_location='cpu',weights_only=True)
        name=Path(record['path']).stem
        for nt in ('EOA','CA'):
            source=graph[nt].x[batch['node_ids'][nt]].cpu()
            require(len(source)>0,'Meta-IFD requires both node types in each sampled batch')
            path=output/f'{name}_{nt}_views.npy'
            values=np.lib.format.open_memmap(path,mode='w+',dtype=np.float32,shape=(views,4,len(source),14))
            for view in range(views):
                interaction=0
                for rid,et in enumerate(graph.edge_types):
                    if et[0]!=nt:continue
                    for start in range(0,len(source),65536):
                        part=source[start:start+65536]
                        z=torch.randn((len(part),50),generator=generator)
                        condition=torch.zeros((len(part),6));condition[:,rid]=1
                        with torch.no_grad():generated=author_row_normalize(decoder(z,part,condition))
                        require(bool(torch.isfinite(generated).all()),'Nonfinite generated view')
                        values[view,interaction,start:start+len(part)]=generated.numpy()
                    interaction+=1
                require(interaction==3,'Expected three outgoing relation types')
                values[view,3]=source.numpy()
            values.flush();values._mmap.close();del values
            files[path.name]=sha256(path)
    return files


## Implementation: models

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: models.imports
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv,SAGEConv
ARM_WIDTHS={'X14':14,'Pure36':36,'X14_Pure36':50,'X14_SIFE44':58}


### arm_columns

Map named arms into source14+SIFE44 moments; Pure36 never selects source14.

In [ ]:
# SIFE20_SOURCE_CELL: models.arm_columns
def arm_columns(arm):
    """Map named arms into source14+SIFE44 moments; Pure36 never selects source14."""
    require(arm in ARM_WIDTHS,'Unknown input arm')
    if arm=='X14':return tuple(range(14))
    extra=tuple(14+i for i in (PURE36_INDICES if arm in ('Pure36','X14_Pure36') else range(44)))
    return extra if arm=='Pure36' else tuple(range(14))+extra


### transform_inputs

Signed-log only source14 for GAT; Pure36 and all extra columns keep their defined scale.

In [ ]:
# SIFE20_SOURCE_CELL: models.transform_inputs
def transform_inputs(values,arm,model_name):
    """Signed-log only source14 for GAT; Pure36 and all extra columns keep their defined scale."""
    require(values.ndim==2 and values.shape[1]==ARM_WIDTHS[arm],'Input width mismatch')
    if model_name!='gat' or arm=='Pure36':return values
    source=values[:,:14]
    source=source.sign()*source.abs().log1p()
    return source if values.shape[1]==14 else torch.cat([source,values[:,14:]],dim=1)


### StudyGAT

Native4-head GAT -> ELU ->4-head GAT2; frozen TRAIN normalization, no dropout/encoder.

In [ ]:
# SIFE20_SOURCE_CELL: models.StudyGAT
class StudyGAT(nn.Module):
    """Native4-head GAT -> ELU ->4-head GAT2; frozen TRAIN normalization, no dropout/encoder."""
    def __init__(self,arm,hidden,statistics):
        super().__init__();self.arm=arm;self.input_width=ARM_WIDTHS[arm]
        columns=list(arm_columns(arm))
        self.register_buffer('input_mean',torch.tensor(statistics['mean'],dtype=torch.float32)[columns])
        self.register_buffer('input_scale',torch.tensor(statistics['scale'],dtype=torch.float32)[columns])
        require(bool(torch.isfinite(self.input_mean).all()) and bool((self.input_scale>0).all()),'Invalid GAT statistics')
        self.conv1=GATConv(self.input_width,hidden,heads=4,concat=True,dropout=0.,negative_slope=.2,add_self_loops=True,bias=True)
        self.conv2=GATConv(4*hidden,2,heads=4,concat=False,dropout=0.,negative_slope=.2,add_self_loops=True,bias=True)
    def forward(self,x,edge_index):
        h=(transform_inputs(x,self.arm,'gat')-self.input_mean)/self.input_scale
        return self.conv2(F.elu(self.conv1(h,edge_index)),edge_index)


### StudySAGE

Raw source14/standardized extras -> Linear/tanh -> projected meanSAGE -> ReLU -> SAGE2.

In [ ]:
# SIFE20_SOURCE_CELL: models.StudySAGE
class StudySAGE(nn.Module):
    """Raw source14/standardized extras -> Linear/tanh -> projected meanSAGE -> ReLU -> SAGE2."""
    def __init__(self,arm,hidden,statistics):
        super().__init__();self.arm=arm;self.input_width=ARM_WIDTHS[arm]
        columns=list(arm_columns(arm));mean=torch.tensor(statistics['mean'],dtype=torch.float32)[columns]
        scale=torch.tensor(statistics['scale'],dtype=torch.float32)[columns]
        # Preserve the reviewed raw-X14 backbone in every arm that has source14.
        if arm!='Pure36':mean[:14]=0.;scale[:14]=1.
        self.register_buffer('input_mean',mean);self.register_buffer('input_scale',scale)
        self.input_encoder=nn.Linear(self.input_width,hidden)
        self.conv1=SAGEConv(hidden,hidden,aggr='mean',project=True,normalize=False,root_weight=True,bias=True)
        self.conv2=SAGEConv(hidden,2,aggr='mean',project=True,normalize=False,root_weight=True,bias=True)
    def forward(self,x,edge_index):
        values=(x-self.input_mean)/self.input_scale
        h=torch.tanh(self.input_encoder(values))
        return self.conv2(F.relu(self.conv1(h,edge_index)),edge_index)


### build_classifier

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: models.build_classifier
def build_classifier(model_name,arm,config,graph,target,statistics):
    if model_name=='gat':return StudyGAT(arm,config['hidden'],statistics)
    if model_name=='sage':return StudySAGE(arm,config['hidden'],statistics)
    require(model_name=='metaifd' and arm!='Pure36','Meta-IFD Pure36-only is unsupported: decoder needs source14')
    return HMSL(config['hidden'],2,graph,config['views'],target,input_width=ARM_WIDTHS[arm])


### lookup_features

Lookup exact full-incident SIFE44 for the sampled global node IDs.

In [ ]:
# SIFE20_SOURCE_CELL: models.lookup_features
def lookup_features(feature_dir,node_type,ids):
    """Lookup exact full-incident SIFE44 for the sampled global node IDs."""
    keys=np.load(Path(feature_dir)/(node_type+'_ids.npy'),mmap_mode='r',allow_pickle=False)
    values=np.load(Path(feature_dir)/(node_type+'_sife44.npy'),mmap_mode='r',allow_pickle=False)
    ids=np.asarray(ids,dtype=np.int64);positions=np.searchsorted(keys,ids)
    require(bool((positions<len(keys)).all()) and np.array_equal(keys[positions],ids),'Missing node feature lookup')
    return np.asarray(values[positions],dtype=np.float64)


### forward_classifier

Use the same sampled topology across all arms; Pure36 does not read source14.

In [ ]:
# SIFE20_SOURCE_CELL: models.forward_classifier
def forward_classifier(model,study,split,arm,batch,role):
    """Use the same sampled topology across all arms; Pure36 does not read source14."""
    p=study['protocol'];graph=study['data']['graph'];device=study['device'];model_name=p['model']
    ids=batch['node_ids'];directory=split['folder']/'cache'/role
    extra_raw={nt:lookup_features(directory/'features',nt,values.numpy()) for nt,values in ids.items()}
    edges={et:ix.to(device) for et,ix in batch['edge_index'].items()}
    if model_name=='metaifd':
        views={nt:torch.from_numpy(np.array(np.load(directory/'views'/f"{batch['batch_name']}_{nt}_views.npy",mmap_mode='r'),copy=True)).to(device)
               for nt in ('EOA','CA')}
        extra=None
        if arm!='X14':
            selected=list(PURE36_INDICES) if arm=='X14_Pure36' else list(range(44))
            mean=np.asarray(split['statistics']['mean'])[14:][selected]
            scale=np.asarray(split['statistics']['scale'])[14:][selected]
            extra={nt:torch.from_numpy(((value[:,selected]-mean)/scale).astype(np.float32)).to(device) for nt,value in extra_raw.items()}
        logits,contrast=model(views,edges,extra)
        return logits[:len(batch['seed_ids'])],contrast
    values=[]
    for nt in ('EOA','CA'):
        source=None if arm=='Pure36' else graph[nt].x[ids[nt]].numpy()
        value=select_inputs(source,extra_raw[nt],arm,model_name)
        values.append(torch.from_numpy(value.astype(np.float32)).to(device))
    offset={'EOA':0,'CA':len(ids['EOA'])}
    homogeneous=[ix+torch.tensor([[offset[et[0]]],[offset[et[2]]]],device=device) for et,ix in edges.items()]
    logits=model(torch.cat(values),torch.cat(homogeneous,dim=1));begin=offset[p['target']]
    return logits[begin:begin+len(batch['seed_ids'])],logits.new_zeros(())


## Implementation: metrics

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: metrics.imports
import numpy as np
from sklearn.metrics import accuracy_score, average_precision_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score


### classification_metrics

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: metrics.classification_metrics
def classification_metrics(labels, scores, predictions):
    labels, scores, predictions = np.asarray(labels), np.asarray(scores), np.asarray(predictions)
    require(np.isfinite(scores).all(), 'Nonfinite scores')
    tn, fp, fn, tp = confusion_matrix(labels, predictions, labels=[0,1]).ravel()
    return dict(macro_f1=float(f1_score(labels, predictions, average='macro', labels=[0,1], zero_division=0)),
                micro_f1=float(f1_score(labels, predictions, average='micro', zero_division=0)),
                binary_f1=float(f1_score(labels, predictions, zero_division=0)),
                precision=float(precision_score(labels, predictions, zero_division=0)),
                recall=float(recall_score(labels, predictions, zero_division=0)),
                accuracy=float(accuracy_score(labels, predictions)), fpr=float(fp/(fp+tn)) if fp+tn else None,
                average_precision=float(average_precision_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
                roc_auc=float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else None,
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), n=len(labels))


## Implementation: tabular_inputs

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: tabular_inputs.imports
import numpy as np
import pandas as pd
SOURCE14_NAMES=('bal_c','V_inv_c','V_return_c','N_inv_c','N_return_c',
    'V_meaninv_c','V_meanreturn_c','bal_t','V_inv_t','V_return_t','N_inv_t','N_return_t','V_meaninv_t','V_meanreturn_t')
LR_PROFILES=('signed_log1p','raw_reference')


### tabular_column_names

Preserve exact source/extra order; reject mismatched prediction schemas.

In [ ]:
# SIFE20_SOURCE_CELL: tabular_inputs.tabular_column_names
def tabular_column_names(arm):
    """Preserve exact source/extra order; reject mismatched prediction schemas."""
    extra=tuple(SIFE44_NAMES[index] for index in PURE36_INDICES) if arm in ('Pure36','X14_Pure36') else tuple(SIFE44_NAMES)
    if arm=='X14':return SOURCE14_NAMES
    return extra if arm=='Pure36' else SOURCE14_NAMES+extra


### tabular_input_frame

LR signed-log touches only source14; named SIFE/Pure columns remain numerically unchanged.

In [ ]:
# SIFE20_SOURCE_CELL: tabular_inputs.tabular_input_frame
def tabular_input_frame(source14,sife44,arm,model_name,lr_profile='signed_log1p'):
    """LR signed-log touches only source14; named SIFE/Pure columns remain numerically unchanged."""
    require(lr_profile in LR_PROFILES,'Unknown fixed LR preprocessing profile')
    values=select_inputs(None if arm=='Pure36' else source14,sife44,arm,model_name).astype(np.float64,copy=True)
    if model_name=='lr' and arm!='Pure36' and lr_profile=='signed_log1p':
        source=values[:,:14]
        values[:,:14]=np.sign(source)*np.log1p(np.abs(source))
    require(np.isfinite(values).all(),'Nonfinite tabular input after declared preprocessing')
    columns=tabular_column_names(arm)
    require(len(columns)==values.shape[1] and len(set(columns))==len(columns),'Invalid input schema')
    return pd.DataFrame(values,columns=columns)


### check_estimator_columns

Require the prediction frame's named columns to match the trained estimator in order.

In [ ]:
# SIFE20_SOURCE_CELL: tabular_inputs.check_estimator_columns
def check_estimator_columns(estimator,frame):
    """Require the prediction frame's named columns to match the trained estimator in order."""
    if hasattr(estimator,'feature_names_in_'):
        require(tuple(estimator.feature_names_in_)==tuple(frame.columns),'Prediction feature names/order differ from TRAIN')


## Implementation: protocol

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.imports
import hashlib
import itertools
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
MODELS=('metaifd','gat','sage','lightgbm','rf')
TABULAR=('lightgbm','rf')


### fixed_configuration

Predeclared configurations; no grid and no per-arm or per-dataset tuning.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.fixed_configuration
def fixed_configuration(model):
    """Predeclared configurations; no grid and no per-arm or per-dataset tuning."""
    return dict(metaifd=dict(hidden=128,lr=.001,views=3,contrast_weight=.1),
        gat=dict(hidden=32,lr=.01),sage=dict(hidden=32,lr=.01),
        lightgbm=dict(n_estimators=300,lr=.01,num_leaves=31),
        rf=dict(n_estimators=100,max_depth=10))[model].copy()


### supported_arms

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.supported_arms
def supported_arms(model):
    require(model in MODELS,'Only the five agreed models are supported')
    return tuple(arm for arm in ARMS if not (model=='metaifd' and arm=='Pure36'))


### notebook_source_signature

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.notebook_source_signature
def notebook_source_signature():
    shell=get_ipython();require(shell is not None,'Use a normal IPython/Jupyter notebook kernel')
    latest={}
    for source in shell.history_manager.input_hist_raw:
        first=source.splitlines()[0] if source.splitlines() else ''
        prefix='# SIFE20_SOURCE_CELL: '
        if first.startswith(prefix):latest[first[len(prefix):]]=hashlib.sha256(source.encode()).hexdigest()
    require(set(DEFINITION_KEYS)<=set(latest),'Run every definition cell above first')
    return {key:latest[key] for key in DEFINITION_KEYS}


### stratified_parts

Address-disjoint60/20/20 using exactly the same split rule as the reviewed LR study.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.stratified_parts
def stratified_parts(roots,seed):
    """Address-disjoint60/20/20 using exactly the same split rule as the reviewed LR study."""
    positions=np.arange(len(roots['ids']));labels=roots['labels']
    development,test=train_test_split(positions,test_size=.2,random_state=seed,stratify=labels)
    train,validation=train_test_split(development,test_size=.25,random_state=seed,stratify=labels[development])
    parts=dict(train=train,validation=validation,test=test)
    combined=np.concatenate(list(parts.values()))
    require(len(combined)==len(set(combined))==len(labels),'Split coverage or overlap error')
    for a,b in [('train','validation'),('train','test'),('validation','test')]:
        require(not set(roots['addresses'][parts[a]])&set(roots['addresses'][parts[b]]),'Address overlap across roles')
    require(all(set(labels[pos])=={0,1} for pos in parts.values()),'Every role must include both classes')
    return parts


### open_study

Freeze all scientific choices, data, seeds, versions and implementation identity.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.open_study
def open_study(data,output,dataset,model,device,source_signature,decoder_reference=None,
               run_mode='full',cpu_threads=2,lr_profile='signed_log1p'):
    """Freeze all scientific choices, data, seeds, versions and implementation identity."""
    require(model in MODELS and dataset in ('Phish','Ponzi'),'Unknown model/dataset')
    require(run_mode in ('full','smoke'),'Full or smoke mode required')
    require(lr_profile in LR_PROFILES,'Unknown fixed LR preprocessing profile')
    require(len(set(data['roots']['addresses']))==len(data['roots']['ids']),'Strict cohort contains duplicates')
    torch.set_num_threads(cpu_threads);neural=model not in TABULAR
    require(neural or str(device)=='cpu','Tabular models run on CPU')
    protocol=dict(study='sife20_native_five_fixed_v1',dataset=dataset,model=model,run_mode=run_mode,
        target='EOA' if dataset=='Phish' else 'CA',feature_version=FEATURE_VERSION,
        arms={arm:ARM_WIDTHS[arm] for arm in supported_arms(model)},
        unsupported_arms={'Pure36':'public conditional decoder requires original source14'} if model=='metaifd' else {},
        split_seeds=list(range(2001,2021)) if run_mode=='full' else [2001],
        model_seeds=list(range(12,17)) if run_mode=='full' else [12],
        split_rule='strict unique-address cohort; stratified60/20/20; no released-mask score pooling',
        config=fixed_configuration(model),configuration_origin='fixed accepted-study budget; not older split-specific validation winners',
        hyperparameter_search=False,primary_metric='Macro-F1',decision='class1 probability>0.5; exacttieclass0; no threshold tuning',
        backbone={'gat':'native4head GAT ELU GAT2; no encoder/BN/dropout',
                  'sage':'Linear/tanh; projected mean SAGEConv h->h->2; no BN/dropout',
                  'metaifd':'public HMSL with optional extras appended after frozen14dim decoder'}.get(model,'standard tabular classifier'),
        lr_source14_profile=lr_profile if model=='lr' else None,
        preprocessing={'lr':('source14 signed-log1p; extras unchanged; TRAIN-root StandardScaler' if lr_profile=='signed_log1p' else
                            'rawsource14 reference; extras unchanged; TRAIN-root StandardScaler'),
            'rf':'rawsource14 and feature-defined extras; no scaler',
            'lightgbm':'rawsource14 and feature-defined extras; no scaler',
            'gat':'signed-log only source14; all active columns z-score from unique typed TRAIN sampled nodes; Pure36 has no source14',
            'sage':'rawsource14 unchanged; extras z-score from unique typed TRAIN sampled nodes before shared Linear/tanh',
            'metaifd':'raw14view generation; extras TRAIN-zscore concatenated algebraically after generation'}[model],
        max_epochs=(1000 if run_mode=='full' else 2) if neural else None,patience=50 if neural else None,
        checkpoint_rule='latest validation Macro-F1 tie; stop stale>50' if neural else 'one fit at fixed budget',
        weight_decay=.0005 if neural else None,batch_size=512 if run_mode=='full' else 8,
        fanouts=[100,100] if run_mode=='full' else [2,2],
        sampler_seed_rule='1201/1202/2301 +10000*(split_seed-2000); actual PyG directional2hop, shared across arms/seeds',
        feature_scope='exact full-incident typed graph records for roots and sampled neighbors; no labels/source14 in SIFE construction',
        test_policy='lock all arms and model seeds on every requested partition before any test sampling or scoring',
        seed_averaging='mean model seeds within split, then mean/sampleSD across20split means; do not treat100runs as independent',
        setting='static transductive; held-out node topology/features may be neighbors, labels not used as model input',
        limitations='exploratory reused corpus, no temporal/external transfer validation; Pure36 is scale-invariant, not money-information-free',
        parameter_budget='input width changes parameter count; report per-arm counts, not capacity matched',
        author_commit='439c6daa936f67f19486be76642e690a2153d0c5',
        meta_loss='public unsquaredL2 TripletMarginLoss mean margin.3; differs from paper equation',
        meta_runner='validation-best loading correction, explicit CPU view RNG and sampling seeds; not exact paper-table recovery',
        tabular_schema='named source14/SIFE DataFrames for fit and predict; exact column-order verification',
        development_provenance='five-model scope selected after earlier results were viewed; historical LR retained outside main table; reused benchmark corpus, not a newly blinded study')
    identity=dict(protocol=protocol,data_manifest=data['manifest'],source_signature=source_signature,
        runtime=runtime_information(device),decoder_reference_hash=json_hash(decoder_reference) if decoder_reference else None)
    output=Path(output);output.mkdir(parents=True,exist_ok=True);path=output/'protocol.json'
    if path.exists():require(read_json(path)==identity,'Inputs/code/config/runtime changed; use a new output directory')
    else:write_json(path,identity)
    return dict(data=data,protocol=protocol,identity=json_hash(identity),output=output,
                device=torch.device(device),decoder_reference=decoder_reference)


### open_split

Persist exact IDs/addresses/labels for each role; same membership for every arm/model.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.open_split
def open_split(study,split_seed):
    """Persist exact IDs/addresses/labels for each role; same membership for every arm/model."""
    require(split_seed in study['protocol']['split_seeds'],'Unknown split seed')
    roots=study['data']['roots'];parts=stratified_parts(roots,split_seed)
    folder=study['output']/f'split_{split_seed}';folder.mkdir(exist_ok=True)
    membership={role:pd.DataFrame(dict(node_id=roots['ids'][pos],address=roots['addresses'][pos],
                label=roots['labels'][pos])) for role,pos in parts.items()}
    identity=json_hash(dict(study=study['identity'],split_seed=split_seed,
        membership={r:f.to_dict('records') for r,f in membership.items()}))
    record=dict(identity=identity,split_seed=split_seed,membership={r:f.to_dict('records') for r,f in membership.items()},
        sizes={r:len(f) for r,f in membership.items()},positives={r:int(f.label.sum()) for r,f in membership.items()})
    path=folder/'membership.json'
    if path.exists():require(read_json(path)==record,'Membership changed')
    else:
        write_json(path,record)
        for role,frame in membership.items():frame.to_csv(folder/(role+'_membership.csv'),index=False)
    return dict(folder=folder,seed=split_seed,parts=parts,membership=membership,identity=identity)


### verify_split_lock

Check coverage and hashes for every arm/seed before evaluation or completed-fit reuse.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.verify_split_lock
def verify_split_lock(study,split):
    """Check coverage and hashes for every arm/seed before evaluation or completed-fit reuse."""
    lock=read_json(split['folder']/'lock.json')
    require(lock['identity']==split['identity'],'Split lock identity changed')
    expected=set(itertools.product(study['protocol']['model_seeds'],study['protocol']['arms']))
    require({(row['model_seed'],row['arm']) for row in lock['fits']}==expected and len(lock['fits'])==len(expected),
            'Missing/duplicated arm or seed')
    verify_files(split['folder'],lock['files'])
    for role in ('train','validation'):
        root=split['folder']/'cache'/role;verify_files(root,read_json(root/'manifest.json')['files'])
    return lock


### verify_study_lock

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: protocol.verify_study_lock
def verify_study_lock(study):
    path=study['output']/'study_lock.json';require(path.exists(),'Fit and freeze every requested partition before TEST')
    lock=read_json(path);require(lock['identity']==study['identity'],'Whole-study identity changed')
    require(set(lock['split_lock_hashes'])=={str(seed) for seed in study['protocol']['split_seeds']},'Missing partition lock')
    for seed in study['protocol']['split_seeds']:
        split=open_split(study,seed)
        require(sha256(split['folder']/'lock.json')==lock['split_lock_hashes'][str(seed)],'Partition lock changed')
        verify_split_lock(study,split)
    return lock


## Implementation: cache

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: cache.imports
from pathlib import Path
import numpy as np
import torch


### prepare_role

Materialize a role once; TEST is gated by all-partition fit locks.

In [ ]:
# SIFE20_SOURCE_CELL: cache.prepare_role
def prepare_role(study,split,role):
    """Materialize a role once; TEST is gated by all-partition fit locks."""
    require(role in ('train','validation','test'),'Unknown role')
    if role=='test':
        path=study['output']/'study_lock.json'
        require(path.exists(),'TEST cache is forbidden before the whole-study fit lock')
        lock=read_json(path)
        require(lock['identity']==study['identity'] and
                lock['split_lock_hashes'].get(str(split['seed']))==sha256(split['folder']/'lock.json'),
                'Test gating lock mismatch')
    root=split['folder']/'cache'/role;manifest_path=root/'manifest.json'
    if manifest_path.exists():
        manifest=read_json(manifest_path);require(manifest['identity']==split['identity'],'Cache identity changed')
        verify_files(root,manifest['files'])
        return [dict(path=str(root/'roots.npz'))] if study['protocol']['model'] in TABULAR else load_role_batches(root/'batches')
    root.mkdir(parents=True,exist_ok=True);p=study['protocol'];data=study['data']
    if p['model'] in TABULAR:
        roots=data['roots'];pos=split['parts'][role]
        np.savez_compressed(root/'roots.npz',ids=roots['ids'][pos],labels=roots['labels'][pos],
                            source14=roots['source14'][pos],sife44=roots['sife44'][pos])
        records=[dict(path=str(root/'roots.npz'))]
    else:
        seed=dict(train=1201,validation=1202,test=2301)[role]+10000*(split['seed']-2000)
        print('Cache native graph:',p['dataset'],p['model'],split['seed'],role,flush=True)
        records=save_pyg_role_batches(data['graph'],split['membership'][role],p['target'],role,
                                     root/'batches',seed,p['fanouts'],p['batch_size'])
        feature_dir=root/'features';feature_dir.mkdir(exist_ok=True)
        by_type={nt:[] for nt in ('EOA','CA')}
        for row in records:
            batch=torch.load(row['path'],map_location='cpu',weights_only=True)
            for nt in by_type:by_type[nt].append(batch['node_ids'][nt].numpy())
        for nt,parts in by_type.items():
            ids=np.unique(np.concatenate(parts))
            print('Exact incident features:',nt,len(ids),'nodes',flush=True)
            values=(extract_sife44(data['node_counts'],data['edges'],nt,ids)['sife44'] if len(ids) else np.empty((0,44)))
            np.save(feature_dir/(nt+'_ids.npy'),ids)
            np.save(feature_dir/(nt+'_sife44.npy'),values)
        if p['model']=='metaifd':
            decoder=materialize_decoder(study['output']/'reference',study['decoder_reference'])
            save_meta_views(data['graph'],records,root/'views',decoder,seed,p['config']['views'])
            del decoder
    files=[path for path in root.rglob('*') if path.is_file() and not path.name.endswith('.partial')]
    write_json(manifest_path,dict(identity=split['identity'],role=role,files=seal_files(root,files),
        feature_scope='full incident graph records for unique role nodes; labels never enter feature construction'))
    return records


### fit_training_statistics

Two-pass float64 moments on unique typed TRAIN neighbors; source columns are independent of extras.

In [ ]:
# SIFE20_SOURCE_CELL: cache.fit_training_statistics
def fit_training_statistics(study,split):
    """Two-pass float64 moments on unique typed TRAIN neighbors; source columns are independent of extras."""
    data=study['data'];root=split['folder']/'cache/train/features'
    total=np.zeros(58,dtype=np.float64);count=0;typed_counts={}
    for nt in ('EOA','CA'):
        ids=np.load(root/(nt+'_ids.npy'),mmap_mode='r',allow_pickle=False)
        extra=np.load(root/(nt+'_sife44.npy'),mmap_mode='r',allow_pickle=False)
        typed_counts[nt]=len(ids);count+=len(ids)
        for start in range(0,len(ids),100000):
            index=torch.tensor(np.array(ids[start:start+100000],copy=True),dtype=torch.long)
            source=data['graph'][nt].x[index].numpy().astype(np.float64)
            if study['protocol']['model']=='gat':source=np.sign(source)*np.log1p(np.abs(source))
            total[:14]+=source.sum(0);total[14:]+=extra[start:start+100000].sum(0)
    require(count>0,'No TRAIN nodes to fit preprocessing')
    mean=total/count;squared=np.zeros(58,dtype=np.float64)
    for nt in ('EOA','CA'):
        ids=np.load(root/(nt+'_ids.npy'),mmap_mode='r',allow_pickle=False)
        extra=np.load(root/(nt+'_sife44.npy'),mmap_mode='r',allow_pickle=False)
        for start in range(0,len(ids),100000):
            index=torch.tensor(np.array(ids[start:start+100000],copy=True),dtype=torch.long)
            source=data['graph'][nt].x[index].numpy().astype(np.float64)
            if study['protocol']['model']=='gat':source=np.sign(source)*np.log1p(np.abs(source))
            squared[:14]+=((source-mean[:14])**2).sum(0)
            squared[14:]+=((extra[start:start+100000]-mean[14:])**2).sum(0)
    scale=np.sqrt(squared/count);constant=scale<1e-12;scale[constant]=1.
    require(np.isfinite(mean).all() and np.isfinite(scale).all(),'Nonfinite TRAIN preprocessing')
    # One float32 representation is also used for Meta/SAGE extras, matching model input precision.
    statistics=dict(mean=mean.astype(np.float32).tolist(),scale=scale.astype(np.float32).tolist(),
        unique_typed_train_nodes=count,counts_by_type=typed_counts,labels_used=False,
        scope='unique typed nodes in TRAIN sampled cache; transductive unlabeled neighbors included',
        constant_columns=np.flatnonzero(constant).tolist(),population_sd=True,
        source_transform='signed_log1p' if study['protocol']['model']=='gat' else 'identity',
        extras_transform='identity before frozen TRAIN z-score; Pure36 selects only its36 columns')
    path=split['folder']/'input_statistics.json'
    if path.exists():require(read_json(path)==statistics,'TRAIN statistics changed')
    else:write_json(path,statistics)
    split['statistics']=statistics
    return statistics


## Implementation: training

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: training.imports
import gc
from pathlib import Path
import time
import warnings
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from threadpoolctl import threadpool_limits


### make_estimator

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: training.make_estimator
def make_estimator(model_name,config,seed,threads):
    if model_name=='lr':
        return Pipeline([('scale',StandardScaler()),('classifier',LogisticRegression(C=config['C'],solver=config['solver'],
            max_iter=config['max_iter'],class_weight=None,random_state=seed))])
    if model_name=='rf':
        return RandomForestClassifier(n_estimators=config['n_estimators'],max_depth=config['max_depth'],
            max_features='sqrt',min_samples_leaf=1,class_weight=None,bootstrap=True,random_state=seed,n_jobs=threads)
    require(model_name=='lightgbm','Unknown tabular model')
    from lightgbm import LGBMClassifier
    return LGBMClassifier(objective='binary',n_estimators=config['n_estimators'],learning_rate=config['lr'],
        num_leaves=config['num_leaves'],max_depth=-1,min_child_samples=20,subsample=1.,colsample_bytree=1.,
        class_weight=None,random_state=seed,n_jobs=threads,verbosity=-1)


### tabular_predict

Predict with a saved pipeline; Pure36 passes no source14 to the input selector.

In [ ]:
# SIFE20_SOURCE_CELL: training.tabular_predict
def tabular_predict(estimator,record,arm,model_name,lr_profile='signed_log1p'):
    """Predict with a saved pipeline; Pure36 passes no source14 to the input selector."""
    with np.load(record['path'],allow_pickle=False) as stored:
        x=tabular_input_frame(None if arm=='Pure36' else stored['source14'],stored['sife44'],arm,model_name,lr_profile)
        ids=stored['ids'].copy();labels=stored['labels'].copy()
    require(np.array_equal(estimator.classes_,[0,1]),'Unexpected class order')
    check_estimator_columns(estimator,x)
    with threadpool_limits(limits=torch.get_num_threads()):
        scores=estimator.predict_proba(x,validate_features=True)[:,1] if model_name=='lightgbm' else estimator.predict_proba(x)[:,1]
    predictions=(scores>.5).astype(np.int64)
    return classification_metrics(labels,scores,predictions),pd.DataFrame(dict(node_id=ids,label=labels,score=scores,prediction=predictions))


### neural_epoch

Supervise seed/root labels only; losses use common sampled inputs for every arm.

In [ ]:
# SIFE20_SOURCE_CELL: training.neural_epoch
def neural_epoch(study,split,model,records,role,arm,optimizer=None):
    """Supervise seed/root labels only; losses use common sampled inputs for every arm."""
    model.train(optimizer is not None);rows=[];loss_sum=0.
    for record in records:
        batch=torch.load(record['path'],map_location='cpu',weights_only=True)
        batch['batch_name']=Path(record['path']).stem;y=batch['seed_labels'].to(study['device'])
        with torch.set_grad_enabled(optimizer is not None):
            logits,contrast=forward_classifier(model,study,split,arm,batch,role)
            loss=F.cross_entropy(logits,y)+study['protocol']['config'].get('contrast_weight',0.)*contrast
            require(bool(torch.isfinite(logits).all() and torch.isfinite(loss)),'Nonfinite logits/loss')
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True);loss.backward()
                require(all(v.grad is None or bool(torch.isfinite(v.grad).all()) for v in model.parameters()),'Nonfinite gradients')
                optimizer.step()
        scores=logits.detach().softmax(1)[:,1].cpu().numpy();labels=y.cpu().numpy()
        predictions=(scores>.5).astype(np.int64);loss_sum+=float(loss.detach().cpu())*len(labels)
        rows.extend(dict(node_id=int(i),label=int(label),score=float(score),prediction=int(pred)) for i,label,score,pred in
                    zip(batch['seed_ids'].tolist(),labels,scores,predictions))
    frame=pd.DataFrame(rows);require(len(frame)>0 and frame.node_id.is_unique,'Missing/duplicate root predictions')
    metrics=classification_metrics(frame.label,frame.score,frame.prediction);metrics['loss']=loss_sum/len(frame)
    return metrics,frame


### fit_arm_seed

Fit one arm/seed at the fixed config; replay validation-best before saving the completed marker.

In [ ]:
# SIFE20_SOURCE_CELL: training.fit_arm_seed
def fit_arm_seed(study,split,arm,seed,train_records,validation_records):
    """Fit one arm/seed at the fixed config; replay validation-best before saving the completed marker."""
    p=study['protocol'];require(arm in p['arms'] and seed in p['model_seeds'],'Unsupported arm or seed')
    trial=split['folder']/f'seed_{seed}'/arm;trial.mkdir(parents=True,exist_ok=True)
    identity=json_hash(dict(split=split['identity'],arm=arm,seed=seed,statistics=split.get('statistics')))
    path=trial/'fit.json'
    if path.exists():
        fit=read_json(path);require(fit['identity']==identity,'Fit belongs to another data/config/preprocessing context')
        verify_files(trial,fit['files']);return fit
    seed_everything(seed);started=time.perf_counter();history=[];captured=[]
    model_name=p['model'];parameter_count=None;detail={};best_epoch=None;stopped_epoch=None
    if model_name in TABULAR:
        with np.load(train_records[0]['path'],allow_pickle=False) as stored:
            x=tabular_input_frame(None if arm=='Pure36' else stored['source14'],stored['sife44'],arm,model_name,p['lr_source14_profile'] or 'signed_log1p')
            labels=stored['labels']
        estimator=make_estimator(model_name,p['config'],seed,torch.get_num_threads())
        with warnings.catch_warnings(record=True) as captured,threadpool_limits(limits=torch.get_num_threads()):
            warnings.simplefilter('always');estimator.fit(x,labels)
        validation,_=tabular_predict(estimator,validation_records[0],arm,model_name,p['lr_source14_profile'] or 'signed_log1p')
        training_metrics,training_predictions=tabular_predict(estimator,train_records[0],arm,model_name,p['lr_source14_profile'] or 'signed_log1p')
        single_class=bool(training_predictions.prediction.nunique()<2)
        checkpoint=trial/'model.joblib';temporary=checkpoint.with_suffix('.partial')
        joblib.dump(estimator,temporary);temporary.replace(checkpoint)
        detail=dict(train_roots=len(labels),input_width=x.shape[1],
            scaler_fit_rows=int(estimator['scale'].n_samples_seen_) if model_name=='lr' else None,
            n_iter=np.asarray(estimator['classifier'].n_iter_).tolist() if model_name=='lr' else None,
            lr_source14_profile=p['lr_source14_profile'],feature_names=list(x.columns),
            single_class_train_prediction=single_class,train_metrics=training_metrics,
            train_predicted_positives=int(training_predictions.prediction.sum()),
            validation_predicted_positives=int(validation['tp']+validation['fp']),
            estimator_input_contract='tabular_input_frame(profile) first, then the saved scaler/classifier; do not feed raw values directly to a signed-log fit')
        if single_class:
            print('DIAGNOSTIC:',p['dataset'],model_name,split['seed'],arm,seed,
                  'TRAIN predictions contain one class; recorded in fit.json. Config/profile/threshold are not changed automatically.',flush=True)
        history=[dict(stage='one_fixed_budget_fit',validation_macro_f1=validation['macro_f1'])];files=[checkpoint]
        del estimator
    else:
        require(split.get('statistics') is not None,'TRAIN preprocessing must be frozen first')
        model=build_classifier(model_name,arm,p['config'],study['data']['graph'],p['target'],split['statistics']).to(study['device'])
        if model_name=='metaifd':model.reset_parameters()
        optimizer=torch.optim.Adam(model.parameters(),lr=p['config']['lr'],weight_decay=p['weight_decay'])
        best=-1.;stale=0;state=None
        for epoch in range(1,p['max_epochs']+1):
            training,_=neural_epoch(study,split,model,train_records,'train',arm,optimizer)
            validation,_=neural_epoch(study,split,model,validation_records,'validation',arm)
            if validation['macro_f1']>=best:
                best=validation['macro_f1'];best_epoch=epoch;stale=0
                state={key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
            else:stale+=1
            history.append(dict(epoch=epoch,train_loss=training['loss'],validation_loss=validation['loss'],
                                validation_macro_f1=validation['macro_f1'],best_epoch=best_epoch,stale=stale))
            if epoch==1 or epoch%25==0:
                print(p['dataset'],model_name,'split',split['seed'],arm,'seed',seed,'epoch',epoch,'val',validation['macro_f1'],flush=True)
            if stale>p['patience']:break
        require(state is not None,'No validation checkpoint selected');stopped_epoch=epoch
        last=trial/'last_checkpoint.pt'
        atomic_torch_save(dict(state_dict={k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                               identity=identity,last_epoch=epoch,diagnostic_only=True),last)
        model.load_state_dict(state,strict=True)
        validation,_=neural_epoch(study,split,model,validation_records,'validation',arm)
        require(abs(validation['macro_f1']-best)<1e-12,'Validation-best checkpoint replay mismatch')
        checkpoint=trial/'checkpoint.pt'
        atomic_torch_save(dict(state_dict=state,identity=identity,best_epoch=best_epoch,
                               arm=arm,statistics=split['statistics']),checkpoint)
        parameter_count=sum(v.numel() for v in model.parameters());files=[checkpoint,last]
        del model,optimizer,state;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()
    pd.DataFrame(history).to_csv(trial/'history.csv',index=False);files.append(trial/'history.csv')
    fit=dict(identity=identity,study_identity=study['identity'],split_identity=split['identity'],split_seed=split['seed'],
        model_seed=seed,arm=arm,n_features=ARM_WIDTHS[arm],config=p['config'],validation_metrics=validation,
        best_epoch=best_epoch,stopped_epoch=stopped_epoch,parameter_count=parameter_count,detail=detail,
        fit_seconds=time.perf_counter()-started,warnings=[str(w.message) for w in captured],
        checkpoint=checkpoint.name,trial=trial.relative_to(study['output']).as_posix(),files=seal_files(trial,files),test_scored=False)
    write_json(path,fit);return fit


### SessionBudgetReached

A resumable stop between fits, not a completed benchmark.

In [ ]:
# SIFE20_SOURCE_CELL: training.SessionBudgetReached
class SessionBudgetReached(RuntimeError):
    """A resumable stop between fits, not a completed benchmark."""
    pass


### fit_split

Used by the explicit stages below; no fitting occurs merely by defining this function.

In [ ]:
# SIFE20_SOURCE_CELL: training.fit_split
def fit_split(study,split,deadline=None):
    if (split['folder']/'lock.json').exists():return verify_split_lock(study,split)
    require(not (study['output']/'study_lock.json').exists(),'Whole-study fitting is closed after lock')
    require(not (split['folder']/'cache/test').exists(),'Test cache appeared before fit lock')
    train=prepare_role(study,split,'train')
    if study['protocol']['model'] not in TABULAR:fit_training_statistics(study,split)
    validation=prepare_role(study,split,'validation');fits=[]
    for seed in study['protocol']['model_seeds']:
        for arm in study['protocol']['arms']:
            if deadline is not None and time.monotonic()>=deadline:
                raise SessionBudgetReached('Session budget reached between fits. Keep WORK_ROOT and rerun unchanged settings to resume. No TEST was scored.')
            fits.append(fit_arm_seed(study,split,arm,seed,train,validation))
            pd.DataFrame([dict(split_seed=r['split_seed'],model_seed=r['model_seed'],arm=r['arm'],
                validation_macro_f1=r['validation_metrics']['macro_f1'],fit_seconds=r['fit_seconds']) for r in fits]).to_csv(split['folder']/'fit_progress.csv',index=False)
    files=[split['folder']/'membership.json',split['folder']/'cache/train/manifest.json',split['folder']/'cache/validation/manifest.json']
    for fit in fits:
        trial=study['output']/fit['trial'];files.extend(trial/name for name in fit['files']);files.append(trial/'fit.json')
    if (split['folder']/'input_statistics.json').exists():files.append(split['folder']/'input_statistics.json')
    lock=dict(identity=split['identity'],fits=fits,files=seal_files(split['folder'],files),test_scored_at_lock=False)
    write_json(split['folder']/'lock.json',lock)
    print('Frozen partition',split['seed'],'fits',len(fits),flush=True)
    return verify_split_lock(study,split)


### fit_all

Complete every requested split/arm/seed before enabling the test evaluation cell.

In [ ]:
# SIFE20_SOURCE_CELL: training.fit_all
def fit_all(study,deadline=None):
    """Complete every requested split/arm/seed before enabling the test evaluation cell."""
    if (study['output']/'study_lock.json').exists():return verify_study_lock(study)
    hashes={}
    for seed in study['protocol']['split_seeds']:
        split=open_split(study,seed);fit_split(study,split,deadline)
        hashes[str(seed)]=sha256(split['folder']/'lock.json')
    lock=dict(identity=study['identity'],split_lock_hashes=hashes,protocol_sha256=sha256(study['output']/'protocol.json'),test_scored_at_lock=False)
    write_json(study['output']/'study_lock.json',lock)
    return verify_study_lock(study)


## Implementation: evaluation

Readable definitions only. Execution starts at the numbered stages below.

In [ ]:
# SIFE20_SOURCE_CELL: evaluation.imports
import gc
from pathlib import Path
import zipfile
import joblib
import numpy as np
import pandas as pd
import torch


### evaluate_trial

Load one fixed checkpoint; no fitting, threshold search or hyperparameter selection.

In [ ]:
# SIFE20_SOURCE_CELL: evaluation.evaluate_trial
def evaluate_trial(study,split,fit,records):
    """Load one fixed checkpoint; no fitting, threshold search or hyperparameter selection."""
    output=study['output'];trial=output/fit['trial'];verify_files(trial,fit['files'])
    marker=trial/'evaluation.json';prediction_path=trial/'predictions.csv'
    global_hash=sha256(output/'study_lock.json')
    if marker.exists():
        saved=read_json(marker)
        require(saved['study_identity']==study['identity'] and saved['study_lock_sha256']==global_hash,'Evaluation identity changed')
        require(sha256(prediction_path)==saved['predictions_sha256'],'Predictions changed')
        return saved['metrics']
    p=study['protocol'];model_name=p['model'];arm=fit['arm']
    if model_name in TABULAR:model=joblib.load(trial/fit['checkpoint'])
    else:
        checkpoint=torch.load(trial/fit['checkpoint'],map_location='cpu',weights_only=True)
        require(checkpoint['identity']==fit['identity'],'Frozen checkpoint identity mismatch')
        model=build_classifier(model_name,arm,p['config'],study['data']['graph'],p['target'],checkpoint['statistics']).to(study['device'])
        model.load_state_dict(checkpoint['state_dict'],strict=True);model.eval()
        split['statistics']=checkpoint['statistics']
    rows=[];frames=[]
    for role in ('train','validation','test'):
        metrics,frame=(tabular_predict(model,records[role][0],arm,model_name,p['lr_source14_profile'] or 'signed_log1p') if model_name in TABULAR else
                       neural_epoch(study,split,model,records[role],role,arm))
        expected=split['membership'][role].set_index('node_id')
        require(frame.node_id.is_unique and set(frame.node_id)==set(expected.index),'Prediction role coverage mismatch')
        require(np.array_equal(frame.label.to_numpy(),expected.loc[frame.node_id,'label'].to_numpy()),'Prediction labels changed')
        if role=='validation':require(abs(metrics['macro_f1']-fit['validation_metrics']['macro_f1'])<1e-12,'Saved validation replay mismatch')
        row=dict(dataset=p['dataset'],model=model_name,split_seed=split['seed'],model_seed=fit['model_seed'],arm=arm,role=role,**metrics)
        rows.append(row)
        frame['address']=expected.loc[frame.node_id,'address'].to_numpy()
        for key in ('dataset','model','split_seed','model_seed','arm','role'):frame[key]=row[key]
        frames.append(frame)
    pd.concat(frames,ignore_index=True).to_csv(prediction_path,index=False)
    write_json(marker,dict(study_identity=study['identity'],study_lock_sha256=global_hash,
        predictions_sha256=sha256(prediction_path),metrics=rows))
    del model;gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return rows


### evaluate_all

Verify every partition lock before the first TEST prediction; completed runs reuse verified artifacts.

In [ ]:
# SIFE20_SOURCE_CELL: evaluation.evaluate_all
def evaluate_all(study):
    """Verify every partition lock before the first TEST prediction; completed runs reuse verified artifacts."""
    lock=verify_study_lock(study);output=study['output']
    require(sha256(output/'protocol.json')==lock['protocol_sha256'],'Frozen protocol changed')
    require(read_json(output/'protocol.json')['runtime']==runtime_information(study['device']),'Runtime changed after fitting')
    if (output/'RESULT_MANIFEST.json').exists():
        result=read_json(output/'RESULT_MANIFEST.json')
        require(result['identity']==study['identity'] and result['study_lock_sha256']==sha256(output/'study_lock.json'),'Result identity changed')
        verify_files(output,result['files'])
        return pd.read_csv(output/'metrics.csv')
    metrics=[]
    for seed in study['protocol']['split_seeds']:
        split=open_split(study,seed);split_lock=verify_split_lock(study,split)
        records={role:prepare_role(study,split,role) for role in ('train','validation','test')}
        for fit in split_lock['fits']:metrics.extend(evaluate_trial(study,split,fit,records))
        pd.DataFrame(metrics).to_csv(output/'evaluation_progress.csv',index=False)
        print('Scored frozen partition',seed,flush=True)
    frame=pd.DataFrame(metrics);frame.to_csv(output/'metrics.csv',index=False)
    summarize(study,frame)
    paths=[output/name for name in ['protocol.json','study_lock.json','metrics.csv','split_means.csv','summary.csv','paired_macro_f1_deltas_pp.csv']]
    paths.extend(output.glob('split_*/seed_*/*/predictions.csv'));paths.extend(output.glob('split_*/seed_*/*/evaluation.json'))
    write_json(output/'RESULT_MANIFEST.json',dict(identity=study['identity'],status='COMPLETE',
        study_lock_sha256=sha256(output/'study_lock.json'),files=seal_files(output,paths),
        full_study=study['protocol']['run_mode']=='full',test_used_for_selection=False))
    return frame


### summarize

Average seeds within each split first; retain every split and paired arm delta.

In [ ]:
# SIFE20_SOURCE_CELL: evaluation.summarize
def summarize(study,metrics):
    """Average seeds within each split first; retain every split and paired arm delta."""
    names=['macro_f1','micro_f1','binary_f1','precision','recall','accuracy','average_precision','roc_auc','fpr']
    test=metrics.loc[metrics.role=='test'];p=study['protocol']
    expected={(seed,model_seed,arm) for seed in p['split_seeds'] for model_seed in p['model_seeds'] for arm in p['arms']}
    require(set(zip(test.split_seed,test.model_seed,test.arm))==expected and len(test)==len(expected),'Incomplete test coverage')
    splits=test.groupby(['dataset','model','arm','split_seed'])[names].mean().reset_index()
    splits.to_csv(study['output']/'split_means.csv',index=False);rows=[]
    for arm,frame in splits.groupby('arm'):
        for key in names:
            values=frame[key].to_numpy()
            rows.append(dict(dataset=p['dataset'],model=p['model'],arm=arm,metric=key,
                mean_percent=float(values.mean()*100),sd_across_split_means_pp=float(values.std(ddof=1)*100) if len(values)>1 else None,
                split_count=len(values),model_seeds_per_split=len(p['model_seeds']),
                inference='descriptive repeated holdout; overlapping corpus; not100independent datasets'))
    summary=pd.DataFrame(rows);summary.to_csv(study['output']/'summary.csv',index=False)
    pivot=splits.pivot(index='split_seed',columns='arm',values='macro_f1')
    deltas=pivot.subtract(pivot['X14'],axis=0).drop(columns='X14')*100
    deltas.to_csv(study['output']/'paired_macro_f1_deltas_pp.csv')
    return summary


### export_results

Export partial progress or complete audit evidence; full snapshots are available for exact resume.

In [ ]:
# SIFE20_SOURCE_CELL: evaluation.export_results
def export_results(study,output,allow_partial=False,include_caches=False):
    """Export partial progress or complete audit evidence; full snapshots are available for exact resume."""
    root=study['output'];complete=(root/'RESULT_MANIFEST.json').exists()
    require(complete or allow_partial,'Evaluation is incomplete; explicitly request a PARTIAL progress export')
    if complete:
        verify_study_lock(study);verify_files(root,read_json(root/'RESULT_MANIFEST.json')['files'])
    output=Path(output)
    with zipfile.ZipFile(output,'w',compression=zipfile.ZIP_DEFLATED) as archive:
        for path in root.rglob('*'):
            if not path.is_file() or path.name.endswith('.partial'):continue
            if path.resolve()==output.resolve():continue
            relative=path.relative_to(root)
            if not include_caches and 'cache' in relative.parts and path.name not in ('manifest.json','index.json'):continue
            archive.write(path,'results/'+study['protocol']['run_mode']+'/'+relative.as_posix())
        data_output=study['data'].get('data_output')
        if data_output:
            for name in ('roots.npz','data_manifest.json'):
                path=Path(data_output)/name
                if path.exists():archive.write(path,'data/'+name)
        archive.writestr('EXPORT_SCOPE.json',__import__('json').dumps(dict(complete=complete,caches_included=include_caches,
            resume='restore full WORK_ROOT including sampled cache files; small audit ZIP alone omits required large cached draws',
            raw_graph_and_ponzi_sidecars_included=False),indent=2))
    return output


## Plain reference data

Audited address mapping,checksums and (Meta-IFD only) pretrained public decoder numeric tensors. This is ordinary data,not encoded code,precomputed SIFE features or trained classifier weights.

In [ ]:
REFERENCE={'dataset': 'Phish',
 'graph_sha256': '8cc653bb389ea1a316f72c3893d0487ab09321552ae022ceaa89cd2499f690e8',
 'ids': [0,
         1,
         2,
         3,
         4,
         5,
         6,
         7,
         8,
         9,
         10,
         11,
         12,
         13,
         14,
         15,
         16,
         17,
         18,
         19,
         20,
         21,
         22,
         23,
         24,
         25,
         26,
         27,
         28,
         29,
         30,
         31,
         32,
         33,
         34,
         35,
         36,
         37,
         38,
         39,
         40,
         41,
         42,
         43,
         44,
         45,
         46,
         47,
         48,
         49,
         50,
         51,
         52,
         53,
         54,
         55,
         56,
         57,
         58,
         59,
         60,
         61,
         62,
         63,
         64,
         65,
         66,
         67,
         68,
         69,
         70,
         71,
         72,
         73,
         74,
         75,
         76,
         77,
         78,
         79,
         80,
         81,
         82,
         83,
         84,
         85,
         86,
         87,
         88,
         89,
         90,
         91,
         92,
         93,
         94,
         95,
         96,
         97,
         98,
         99,
         100,
         101,
         102,
         103,
         104,
         105,
         106,
         107,
         108,
         109,
         110,
         111,
         112,
         113,
         114,
         115,
         116,
         117,
         118,
         119,
         120,
         121,
         122,
         123,
         124,
         125,
         126,
         127,
         128,
         129,
         130,
         131,
         132,
         133,
         134,
         135,
         136,
         137,
         138,
         139,
         140,
         141,
         142,
         143,
         144,
         145,
         146,
         147,
         148,
         149,
         150,
         151,
         152,
         153,
         154,
         155,
         156,
         157,
         158,
         159,
         160,
         161,
         162,
         163,
         164,
         165,
         166,
         167,
         168,
         169,
         170,
         171,
         172,
         173,
         174,
         175,
         176,
         177,
         178,
         179,
         180,
         181,
         182,
         183,
         184,
         185,
         186,
         187,
         188,
         189,
         190,
         191,
         192,
         193,
         194,
         195,
         196,
         197,
         198,
         199,
         200,
         201,
         202,
         203,
         204,
         205,
         206,
         207,
         208,
         209,
         210,
         211,
         212,
         213,
         214,
         215,
         216,
         217,
         218,
         219,
         220,
         221,
         222,
         223,
         224,
         225,
         226,
         227,
         228,
         229,
         230,
         231,
         232,
         233,
         234,
         235,
         236,
         237,
         238,
         239,
         240,
         241,
         242,
         243,
         244,
         245,
         246,
         247,
         248,
         249,
         250,
         251,
         252,
         253,
         254,
         255,
         256,
         257,
         258,
         259,
         260,
         261,
         262,
         263,
         264,
         265,
         266,
         267,
         268,
         269,
         270,
         271,
         272,
         273,
         274,
         275,
         276,
         277,
         278,
         279,
         280,
         281,
         282,
         283,
         284,
         285,
         286,
         287,
         288,
         289,
         290,
         291,
         292,
         293,
         294,
         295,
         296,
         297,
         298,
         299,
         300,
         301,
         302,
         303,
         304,
         305,
         306,
         307,
         308,
         309,
         310,
         311,
         312,
         313,
         314,
         315,
         316,
         317,
         318,
         319,
         320,
         321,
         322,
         323,
         324,
         325,
         326,
         327,
         328,
         329,
         330,
         331,
         332,
         333,
         334,
         335,
         336,
         337,
         338,
         339,
         340,
         341,
         342,
         343,
         344,
         345,
         346,
         347,
         348,
         349,
         350,
         351,
         352,
         353,
         354,
         355,
         356,
         357,
         358,
         359,
         360,
         361,
         362,
         363,
         364,
         365,
         366,
         367,
         368,
         369,
         370,
         371,
         372,
         373,
         374,
         375,
         376,
         377,
         378,
         379,
         380,
         381,
         382,
         383,
         384,
         385,
         386,
         387,
         388,
         389,
         390,
         391,
         392,
         393,
         394,
         395,
         396,
         397,
         398,
         399,
         400,
         401,
         402,
         403,
         404,
         405,
         406,
         407,
         408,
         409,
         410,
         411,
         412,
         413,
         414,
         415,
         416,
         417,
         418,
         419,
         420,
         421,
         422,
         423,
         424,
         425,
         426,
         427,
         428,
         429,
         430,
         431,
         432,
         433,
         434,
         435,
         436,
         437,
         438,
         439,
         440,
         441,
         442,
         443,
         444,
         445,
         446,
         447,
         448,
         449,
         450,
         451,
         452,
         453,
         454,
         455,
         456,
         457,
         458,
         459,
         460,
         461,
         462,
         463,
         464,
         465,
         466,
         467,
         468,
         469,
         470,
         471,
         472,
         473,
         474,
         475,
         476,
         477,
         478,
         479,
         480,
         481,
         482,
         483,
         484,
         485,
         486,
         487,
         488,
         489,
         490,
         491,
         492,
         493,
         494,
         495,
         496,
         497,
         498,
         499,
         500,
         501,
         502,
         503,
         504,
         505,
         506,
         507,
         508,
         509,
         510,
         511,
         512,
         513,
         514,
         515,
         516,
         517,
         518,
         519,
         520,
         521,
         522,
         523,
         524,
         525,
         526,
         527,
         528,
         529,
         530,
         531,
         532,
         533,
         534,
         535,
         536,
         537,
         538,
         539,
         540,
         541,
         542,
         543,
         544,
         545,
         546,
         547,
         548,
         549,
         550,
         551,
         552,
         553,
         554,
         555,
         556,
         557,
         558,
         559,
         560,
         561,
         562,
         563,
         564,
         565,
         566,
         567,
         568,
         569,
         570,
         571,
         572,
         573,
         574,
         575,
         576,
         577,
         578,
         579,
         580,
         581,
         582,
         583,
         584,
         585,
         586,
         587,
         588,
         589,
         590,
         591,
         592,
         593,
         594,
         595,
         596,
         597,
         598,
         599,
         600,
         601,
         602,
         603,
         604,
         605,
         606,
         607,
         608,
         609,
         610,
         611,
         612,
         613,
         614,
         615,
         616,
         617,
         618,
         619,
         620,
         621,
         622,
         623,
         624,
         625,
         626,
         627,
         628,
         629,
         630,
         631,
         632,
         633,
         634,
         635,
         636,
         637,
         638,
         639,
         640,
         641,
         642,
         643,
         644,
         645,
         646,
         647,
         648,
         649,
         650,
         651,
         652,
         653,
         654,
         655,
         656,
         657,
         658,
         659,
         660,
         661,
         662,
         663,
         664,
         665,
         666,
         667,
         668,
         669,
         670,
         671,
         672,
         673,
         674,
         675,
         676,
         677,
         678,
         679,
         680,
         681,
         682,
         683,
         684,
         685,
         686,
         687,
         688,
         689,
         690,
         691,
         692,
         693,
         694,
         695,
         696,
         697,
         698,
         699,
         700,
         701,
         702,
         703,
         704,
         705,
         706,
         707,
         708,
         709,
         710,
         711,
         712,
         713,
         714,
         715,
         716,
         717,
         718,
         719,
         720,
         721,
         722,
         723,
         724,
         725,
         726,
         727,
         728,
         729,
         730,
         731,
         732,
         733,
         734,
         735,
         736,
         737,
         738,
         739,
         740,
         741,
         742,
         743,
         744,
         745,
         746,
         747,
         748,
         749,
         750,
         751,
         752,
         753,
         754,
         755,
         756,
         757,
         758,
         759,
         760,
         761,
         762,
         763,
         764,
         765,
         766,
         767,
         768,
         769,
         770,
         771,
         772,
         773,
         774,
         775,
         776,
         777,
         778,
         779,
         780,
         781,
         782,
         783,
         784,
         785,
         786,
         787,
         788,
         789,
         790,
         791,
         792,
         793,
         794,
         795,
         796,
         797,
         798,
         799,
         800,
         801,
         802,
         803,
         804,
         805,
         806,
         807,
         808,
         809,
         810,
         811,
         812,
         813,
         814,
         815,
         816,
         817,
         818,
         819,
         820,
         821,
         822,
         823,
         824,
         825,
         826,
         827,
         828,
         829,
         830,
         831,
         832,
         833,
         834,
         835,
         836,
         837,
         838,
         839,
         840,
         841,
         842,
         843,
         844,
         845,
         846,
         847,
         848,
         849,
         850,
         851,
         852,
         853,
         854,
         855,
         856,
         857,
         858,
         859,
         860,
         861,
         862,
         863,
         864,
         865,
         866,
         867,
         868,
         869,
         870,
         871,
         872,
         873,
         874,
         875,
         876,
         877,
         878,
         879,
         880,
         881,
         882,
         883,
         884,
         885,
         886,
         887,
         888,
         889,
         890,
         891,
         892,
         893,
         894,
         895,
         896,
         897,
         898,
         899,
         900,
         901,
         902,
         903,
         904,
         905,
         906,
         907,
         908,
         909,
         910,
         911,
         912,
         913,
         914,
         915,
         916,
         917,
         918,
         919,
         920,
         921,
         922,
         923,
         924,
         925,
         926,
         927,
         928,
         929,
         930,
         931,
         932,
         933,
         934,
         935,
         936,
         937,
         938,
         939,
         940,
         941,
         942,
         943,
         944,
         945,
         946,
         947,
         948,
         949,
         950,
         951,
         952,
         953,
         954,
         955,
         956,
         957,
         958,
         959,
         960,
         961,
         962,
         963,
         964,
         965,
         966,
         967,
         968,
         969,
         970,
         971,
         972,
         973,
         974,
         975,
         976,
         977,
         978,
         979,
         980,
         981,
         982,
         983,
         984,
         985,
         986,
         987,
         988,
         989,
         990,
         991,
         992,
         993,
         994,
         995,
         996,
         997,
         998,
         999,
         1000,
         1001,
         1002,
         1003,
         1004,
         1005,
         1006,
         1007,
         1008,
         1009,
         1010,
         1011,
         1012,
         1013,
         1014,
         1015,
         1016,
         1017,
         1018,
         1019,
         1020,
         1021,
         1022,
         1023,
         1024,
         1025,
         1026,
         1027,
         1028,
         1029,
         1030,
         1031,
         1032,
         1033,
         1034,
         1035,
         1036,
         1037,
         1038,
         1039,
         1040,
         1041,
         1042,
         1043,
         1044,
         1045,
         1046,
         1047,
         1048,
         1049,
         1050,
         1051,
         1052,
         1053,
         1054,
         1055,
         1056,
         1057,
         1058,
         1059,
         1060,
         1061,
         1062,
         1063,
         1064,
         1065,
         1066,
         1067,
         1068,
         1069,
         1070,
         1071,
         1072,
         1073,
         1074,
         1075,
         1076,
         1077,
         1078,
         1079,
         1080,
         1081,
         1082,
         1083,
         1084,
         1085,
         1086,
         1087,
         1088,
         1089,
         1090,
         1091,
         1092,
         1093,
         1094,
         1095,
         1096,
         1097,
         1098,
         1099,
         1100,
         1101,
         1102,
         1103,
         1104,
         1105,
         1106,
         1107,
         1108,
         1109,
         1110,
         1111,
         1112,
         1113,
         1114,
         1115,
         1116,
         1117,
         1118,
         1119,
         1120,
         1121,
         1122,
         1123,
         1124,
         1125,
         1126,
         1127,
         1128,
         1129,
         1130,
         1131,
         1132,
         1133,
         1134,
         1135,
         1136,
         1137,
         1138,
         1139,
         1140,
         1141,
         1142,
         1143,
         1144,
         1145,
         1146,
         1147,
         1148,
         1149,
         1150,
         1151,
         1152,
         1153,
         1154,
         1155,
         1156,
         1157,
         1158,
         1159,
         1160,
         1161,
         1162,
         1163,
         1164,
         1165,
         1166,
         1167,
         1168,
         1169,
         1170,
         1171,
         1172,
         1173,
         1174,
         1175,
         1176,
         1177,
         1178,
         1179,
         1180,
         1181,
         1182,
         1183,
         1184,
         1185,
         1186,
         1187,
         1188,
         1189,
         1190,
         1191,
         1192,
         1193,
         1194,
         1195,
         1196,
         1197,
         1198,
         1199,
         1200,
         1201,
         1202,
         1203,
         1204,
         1205,
         1206,
         1207,
         1208,
         1209,
         1210,
         1211,
         1212,
         1213,
         1214,
         1215,
         1216,
         1217,
         1218,
         1219,
         1220,
         1221,
         1222,
         1223,
         1224,
         1225,
         1226,
         1227,
         1228,
         1229,
         1230,
         1231,
         1232,
         1233,
         1234,
         1235,
         1236,
         1237,
         1238,
         1239,
         1240,
         1241,
         1242,
         1243,
         1244,
         1245,
         1246,
         1247,
         1248,
         1249,
         1250,
         1251,
         1252,
         1253,
         1254,
         1255,
         1256,
         1257,
         1258,
         1259,
         1260,
         1261,
         1262,
         1263,
         1264,
         1265,
         1266,
         1267,
         1268,
         1269,
         1270,
         1271,
         1272,
         1273,
         1274,
         1275,
         1276,
         1277,
         1278,
         1279,
         1280,
         1281,
         1282,
         1283,
         1284,
         1285,
         1286,
         1287,
         1288,
         1289,
         1290,
         1291,
         1292,
         1293,
         1294,
         1295,
         1296,
         1297,
         1298,
         1299,
         1300,
         1301,
         1302,
         1303,
         1304,
         1305,
         1306,
         1307,
         1308,
         1309,
         1310,
         1311,
         1312,
         1313,
         1314,
         1315,
         1316,
         1317,
         1318,
         1319,
         1320,
         1321,
         1322,
         1323,
         1324,
         1325,
         1326,
         1327,
         1328,
         1329,
         1330,
         1331,
         1332,
         1333,
         1334,
         1335,
         1336,
         1337,
         1338,
         1339,
         1340,
         1341,
         1342,
         1343,
         1344,
         1345,
         1346,
         1347,
         1348,
         1349,
         1350,
         1351,
         1352,
         1353,
         1354,
         1355,
         1356,
         1357,
         1358,
         1359,
         1360,
         1361,
         1362,
         1363,
         1364,
         1365,
         1366,
         1367,
         1368,
         1369,
         1370,
         1371,
         1372,
         1373,
         1374,
         1375,
         1376,
         1377,
         1378,
         1379,
         1380,
         1381,
         1382,
         1383,
         1384,
         1385,
         1386,
         1387,
         1388,
         1389,
         1390,
         1391,
         1392,
         1393,
         1394,
         1395,
         1396,
         1397,
         1398,
         1399,
         1400,
         1401,
         1402,
         1403,
         1404,
         1405,
         1406,
         1407,
         1408,
         1409,
         1410,
         1411,
         1412,
         1413,
         1414,
         1415,
         1416,
         1417,
         1418,
         1419,
         1420,
         1421,
         1422,
         1423,
         1424,
         1425,
         1426,
         1427,
         1428,
         1429,
         1430,
         1431,
         1432,
         1433,
         1434,
         1435,
         1436,
         1437,
         1438,
         1439,
         1440,
         1441,
         1442,
         1443,
         1444,
         1445,
         1446,
         1447,
         1448,
         1449,
         1450,
         1451,
         1452,
         1453,
         1454,
         1455,
         1456,
         1457,
         1458,
         1459,
         1460,
         1461,
         1462,
         1463,
         1464,
         1465,
         1466,
         1467,
         1468,
         1469,
         1470,
         1471,
         1472,
         1473,
         1474,
         1475,
         1476,
         1477,
         1478,
         1479,
         1480,
         1481,
         1482,
         1483,
         1484,
         1485,
         1486,
         1487,
         1488,
         1489,
         1490,
         1491,
         1492,
         1493,
         1494,
         1495,
         1496,
         1497,
         1498,
         1499,
         1500,
         1501,
         1502,
         1503,
         1504,
         1505,
         1506,
         1507,
         1508,
         1509,
         1510,
         1511,
         1512,
         1513,
         1514,
         1515,
         1516,
         1517,
         1518,
         1519,
         1520,
         1521,
         1522,
         1523,
         1524,
         1525,
         1526,
         1527,
         1528,
         1529,
         1530,
         1531,
         1532,
         1533,
         1534,
         1535,
         1536,
         1537,
         1538,
         1539,
         1540,
         1541,
         1542,
         1543,
         1544,
         1545,
         1546,
         1547,
         1548,
         1549,
         1550,
         1551,
         1552,
         1553,
         1554,
         1555,
         1556,
         1557,
         1558,
         1559,
         1560,
         1561,
         1562,
         1563,
         1564,
         1565,
         1566,
         1567,
         1568,
         1569,
         1570,
         1571,
         1572,
         1573,
         1574,
         1575,
         1576,
         1577,
         1578,
         1579,
         1580,
         1581,
         1582,
         1583,
         1584,
         1585,
         1586,
         1587,
         1588,
         1589,
         1590,
         1591,
         1592,
         1593,
         1594,
         1595,
         1596,
         1597,
         1598,
         1599,
         1600,
         1601,
         1602,
         1603,
         1604,
         1605,
         1606,
         1607,
         1608,
         1609,
         1610,
         1611,
         1612,
         1613,
         1614,
         1615,
         1616,
         1617,
         1618,
         1619,
         1620,
         1621,
         1622,
         1623,
         1624,
         1625,
         1626,
         1627,
         1628,
         1629,
         1630,
         1631,
         1632,
         1633,
         1634,
         1635,
         1636,
         1637,
         1638,
         1639,
         1640,
         1641,
         1642,
         1643,
         1644,
         1645,
         1646,
         1647,
         1648,
         1649,
         1650,
         1651,
         1652,
         1653,
         1654,
         1655,
         1656,
         1657,
         1658,
         1659,
         1660,
         1661,
         1662,
         1663,
         1664,
         1665,
         1666,
         1667,
         1668,
         1669,
         1670,
         1671,
         1672,
         1673,
         1674,
         1675,
         1676,
         1677,
         1678,
         1679,
         1680,
         1681,
         1682,
         1683,
         1684,
         1685,
         1686,
         1687,
         1688,
         1689,
         1690,
         1691,
         1692,
         1693,
         1694,
         1695,
         1696,
         1697,
         1698,
         1699,
         1700,
         1701,
         1702,
         1703,
         1704,
         1705,
         1706,
         1707,
         1708,
         1709,
         1710,
         1711,
         1712,
         1713,
         1714,
         1715,
         1716,
         1717,
         1718,
         1719,
         1720,
         1721,
         1722,
         1723,
         1724,
         1725,
         1726,
         1727,
         1728,
         1729,
         1730,
         1731,
         1732,
         1733,
         1734,
         1735,
         1736,
         1737,
         1738,
         1739,
         1740,
         1741,
         1742,
         1743,
         1744,
         1745,
         1746,
         1747,
         1748,
         1749,
         1750,
         1751,
         1752,
         1753,
         1754,
         1755,
         1756,
         1757,
         1758,
         1759,
         1760,
         1761,
         1762,
         1763,
         1764,
         1765,
         1766,
         1767,
         1768,
         1769,
         1770,
         1771,
         1772,
         1773,
         1774,
         1775,
         1776,
         1777,
         1778,
         1779,
         1780,
         1781,
         1782,
         1783,
         1784,
         1785,
         1786,
         1787,
         1788,
         1789,
         1790,
         1791,
         1792,
         1793,
         1794,
         1795,
         1796,
         1797,
         1798,
         1799,
         1800,
         1801,
         1802,
         1803,
         1804,
         1805,
         1806,
         1807,
         1808,
         1809,
         1810,
         1811,
         1812,
         1813,
         1814,
         1815,
         1816,
         1817,
         1818,
         1819,
         1820,
         1821,
         1822,
         1823,
         1824,
         1825,
         1826,
         1827,
         1828,
         1829,
         1830,
         1831,
         1832,
         1833,
         1834,
         1835,
         1836,
         1837,
         1838,
         1839,
         1840,
         1841,
         1842,
         1843,
         1844,
         1845,
         1846,
         1847,
         1848,
         1849,
         1850,
         1851,
         1852,
         1853,
         1854,
         1855,
         1856,
         1857,
         1858,
         1859,
         1860,
         1861,
         1862,
         1863,
         1864,
         1865,
         1866,
         1867,
         1868,
         1869,
         1870,
         1871,
         1872,
         1873,
         1874,
         1875,
         1876,
         1877,
         1878,
         1879,
         1880,
         1881,
         1882,
         1883,
         1884,
         1885,
         1886,
         1887,
         1888,
         1889,
         1890,
         1891,
         1892,
         1893,
         1894,
         1895,
         1896,
         1897,
         1898,
         1899,
         1900,
         1901,
         1902,
         1903,
         1904,
         1905,
         1906,
         1907,
         1908,
         1909,
         1910,
         1911,
         1912,
         1913,
         1914,
         1915,
         1916,
         1917,
         1918,
         1919,
         1920,
         1921,
         1922,
         1923,
         1924,
         1925,
         1926,
         1927,
         1928,
         1929,
         1930,
         1931,
         1932,
         1933,
         1934,
         1935,
         1936,
         1937,
         1938,
         1939,
         1940,
         1941,
         1942,
         1943,
         1944,
         1945,
         1946,
         1947,
         1948,
         1949,
         1950,
         1951,
         1952,
         1953,
         1954,
         1955,
         1956,
         1957,
         1958,
         1959,
         1960,
         1961,
         1962,
         1963,
         1964,
         1965,
         1966,
         1967,
         1968,
         1969,
         1970,
         1971,
         1972,
         1973,
         1974,
         1975,
         1976,
         1977,
         1978,
         1979,
         1980,
         1981,
         1982,
         1983,
         1984,
         1985,
         1986,
         1987,
         1988,
         1989,
         1990,
         1991,
         1992,
         1993,
         1994,
         1995,
         1996,
         1997,
         1998,
         1999,
         2000,
         2001,
         2002,
         2003,
         2004,
         2005,
         2006,
         2007,
         2008,
         2009,
         2010,
         2011,
         2012,
         2013,
         2014,
         2015,
         2016,
         2017,
         2018,
         2019,
         2020,
         2021,
         2022,
         2023,
         2024,
         2025,
         2026,
         2027,
         2028,
         2029,
         2030,
         2031,
         2032,
         2033,
         2034,
         2035,
         2036,
         2037,
         2038,
         2039,
         2040,
         2041,
         2042,
         2043,
         2044,
         2045,
         2046,
         2047,
         2048,
         2049,
         2050,
         2051,
         2052,
         2053,
         2054,
         2055,
         2056,
         2057,
         2058,
         2059,
         2060,
         2061,
         2062,
         2063,
         2064,
         2065,
         2066,
         2067,
         2068,
         2069,
         2070,
         2071,
         2072,
         2073,
         2074,
         2075,
         2076,
         2077,
         2078,
         2079,
         2080,
         2081,
         2082,
         2083,
         2084,
         2085,
         2086,
         2087,
         2088,
         2089,
         2090,
         2091,
         2092,
         2093,
         2094,
         2095,
         2096,
         2097,
         2098,
         2099,
         2100,
         2101,
         2102,
         2103,
         2104,
         2105,
         2106,
         2107,
         2108,
         2109,
         2110,
         2111,
         2112,
         2113,
         2114,
         2115,
         2116,
         2117,
         2118,
         2119,
         2120,
         2121,
         2122,
         2123,
         2124,
         2125,
         2126,
         2127,
         2128,
         2129,
         2130,
         2131,
         2132,
         2133,
         2134,
         2135,
         2136,
         2137,
         2138,
         2139,
         2140,
         2141,
         2142,
         2143,
         2144,
         2145,
         2146,
         2147,
         2148,
         2149,
         2150,
         2151,
         2152,
         2153,
         2154,
         2155,
         2156,
         2157,
         2158,
         2159,
         2160,
         2161,
         2162,
         2163,
         2164,
         2165,
         2166,
         2167,
         2168,
         2169,
         2170,
         2171,
         2172,
         2173,
         2174,
         2175,
         2176,
         2177,
         2178,
         2179,
         2180,
         2181,
         2182,
         2183,
         2184,
         2185,
         2186,
         2187,
         2188,
         2189,
         2190,
         2191,
         2192,
         2193,
         2194,
         2195,
         2196,
         2197,
         2198,
         2199,
         2200,
         2201,
         2202,
         2203,
         2204,
         2205,
         2206,
         2207,
         2208,
         2209,
         2210,
         2211,
         2212,
         2213,
         2214,
         2215,
         2216,
         2217,
         2218,
         2219,
         2220,
         2221,
         2222,
         2223,
         2224,
         2225,
         2226,
         2227,
         2228,
         2229,
         2230,
         2231,
         2232,
         2233,
         2234,
         2235,
         2236,
         2237,
         2238,
         2239,
         2240,
         2241,
         2242,
         2243,
         2244,
         2245,
         2246,
         2247,
         2248,
         2249,
         2250,
         2251,
         2252,
         2253,
         2254,
         2255,
         2256,
         2257,
         2258,
         2259,
         2260,
         2261,
         2262,
         2263,
         2264,
         2265,
         2266,
         2267,
         2268,
         2269,
         2270,
         2271,
         2272,
         2273,
         2274,
         2275,
         2276,
         2277,
         2278,
         2279,
         2280,
         2281,
         2282,
         2283,
         2284,
         2285,
         2286,
         2287,
         2288,
         2289,
         2290,
         2291,
         2292,
         2293,
         2294,
         2295,
         2296,
         2297,
         2298,
         2299,
         2300,
         2301,
         2302,
         2303,
         2304,
         2305,
         2306,
         2307,
         2308,
         2309,
         2310,
         2311,
         2312,
         2313,
         2314,
         2315,
         2316,
         2317,
         2318,
         2319,
         2320,
         2321,
         2322,
         2323,
         2324,
         2325,
         2326,
         2327,
         2328,
         2329,
         2330,
         2331,
         2332,
         2333,
         2334,
         2335,
         2336,
         2337,
         2338,
         2339,
         2340,
         2341,
         2342,
         2343,
         2344,
         2345,
         2346,
         2347,
         2348,
         2349,
         2350,
         2351,
         2352,
         2353,
         2354,
         2355,
         2356,
         2357,
         2358,
         2359,
         2360,
         2361,
         2362,
         2363,
         2364,
         2365,
         2366,
         2367,
         2368,
         2369,
         2370,
         2371,
         2372,
         2373,
         2374,
         2375,
         2376,
         2377,
         2378,
         2379,
         2380,
         2381,
         2382,
         2383,
         2384,
         2385,
         2386,
         2387,
         2388,
         2389,
         2390,
         2391,
         2392,
         2393,
         2394,
         2395,
         2396,
         2397,
         2398,
         2399,
         2400,
         2401,
         2402,
         2403,
         2404,
         2405,
         2406,
         2407,
         2408,
         2409,
         2410,
         2411,
         2412,
         2413,
         2414,
         2415,
         2416,
         2417,
         2418,
         2419,
         2420,
         2421,
         2422,
         2423,
         2424,
         2425,
         2426,
         2427,
         2428,
         2429,
         2430,
         2431,
         2432,
         2433,
         2434,
         2435,
         2436,
         2437,
         2438,
         2439,
         2440,
         2441,
         2442,
         2443,
         2444,
         2445,
         2446,
         2447,
         2448,
         2449,
         2450,
         2451,
         2452,
         2453,
         2454,
         2455,
         2456,
         2457,
         2458,
         2459,
         2460,
         2461,
         2462,
         2463,
         2464,
         2465,
         2466,
         2467,
         2468,
         2469,
         2470,
         2471,
         2472,
         2473,
         2474,
         2475,
         2476,
         2477,
         2478,
         2479,
         2480,
         2481,
         2482,
         2483,
         2484,
         2485,
         2486,
         2487,
         2488,
         2489,
         2490,
         2491,
         2492,
         2493,
         2494,
         2495,
         2496,
         2497,
         2498,
         2499,
         2500,
         2501,
         2502,
         2503,
         2504,
         2505,
         2506,
         2507,
         2508,
         2509,
         2510,
         2511,
         2512,
         2513,
         2514,
         2515,
         2516,
         2517,
         2518,
         2519,
         2520,
         2521,
         2522,
         2523,
         2524,
         2525,
         2526,
         2527,
         2528,
         2529,
         2530,
         2531,
         2532,
         2533,
         2534,
         2535,
         2536,
         2537,
         2538,
         2539,
         2540,
         2541,
         2542,
         2543,
         2544,
         2545,
         2546,
         2547,
         2548,
         2549,
         2550,
         2551,
         2552,
         2553,
         2554,
         2555,
         2556,
         2557,
         2558,
         2559,
         2560,
         2561,
         2562,
         2563,
         2564,
         2565,
         2566,
         2567,
         2568,
         2569,
         2570,
         2571,
         2572,
         2573,
         2574,
         2575,
         2576,
         2577,
         2578,
         2579,
         2580,
         2581,
         2582,
         2583,
         2584,
         2585,
         2586,
         2587,
         2588,
         2589,
         2590,
         2591,
         2592,
         2593,
         2594,
         2595,
         2596,
         2597,
         2598,
         2599,
         2600,
         2601,
         2602,
         2603,
         2604,
         2605,
         2606,
         2607,
         2608,
         2609,
         2610,
         2611,
         2612,
         2613,
         2614,
         2615,
         2616,
         2617,
         2618,
         2619,
         2620,
         2621,
         2622,
         2623,
         2624,
         2625,
         2626,
         2627,
         2628,
         2629,
         2630,
         2631,
         2632,
         2633,
         2634,
         2635,
         2636,
         2637,
         2638,
         2639,
         2640,
         2641,
         2642,
         2643,
         2644,
         2645,
         2646,
         2647,
         2648,
         2649,
         2650,
         2651,
         2652,
         2653,
         2654,
         2655,
         2656,
         2657,
         2658,
         2659,
         2660,
         2661,
         2662,
         2663,
         2664,
         2665,
         2666,
         2667,
         2668,
         2669,
         2670,
         2671,
         2672,
         2673,
         2674,
         2675,
         2676,
         2677,
         2678,
         2679,
         2680,
         2681,
         2682,
         2683,
         2684,
         2685,
         2686,
         2687,
         2688,
         2689,
         2690,
         2691,
         2692,
         2693,
         2694,
         2695,
         2696,
         2697,
         2698,
         2699,
         2700,
         2701,
         2702,
         2703,
         2704,
         2705,
         2706,
         2707,
         2708,
         2709,
         2710,
         2711,
         2712,
         2713,
         2714,
         2715,
         2716,
         2717,
         2718,
         2719,
         2720,
         2721,
         2722,
         2723,
         2724,
         2725,
         2726,
         2727,
         2728,
         2729,
         2730,
         2731,
         2732,
         2733,
         2734,
         2735,
         2736,
         2737,
         2738,
         2739,
         2740,
         2741,
         2742,
         2743,
         2744,
         2745,
         2746,
         2747,
         2748,
         2749,
         2750,
         2751,
         2752,
         2753,
         2754,
         2755,
         2756,
         2757,
         2758,
         2759,
         2760,
         2761,
         2762],
 'labels': [1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            1,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0],
 'addresses': ['...'],
 'mapping_asset_sha256': '00adf2f48e03c6bcd69ec39738e7e40443b554249f8955759f577584c78550fc',
 'raw_members': {'Meta-IFD data/Ponzi_unbalance/CA_labeled.csv': {'sha256': 'd9deb51acc377d2a7fc1c04656f0d90a640596684d1cdc14735a0a54be9d85bb',
                                                                  'bytes': 240033},
                 'Meta-IFD data/Ponzi_unbalance/CA_unlabeled.csv': {'sha256': 'a2a2ff2b3a4486ba1b457579b374c6307f0a8654173ac1b63c9475711eb624ef',
                                                                    'bytes': 343178752},
                 'Meta-IFD data/Ponzi_unbalance/EOA.csv': {'sha256': '5299ec5d2fe62aaa1d6347416439c95dc044066bd2dc69c735f1282558a3efef',
                                                           'bytes': 1209674221},
                 'Meta-IFD data/Ponzi_unbalance/edge/call_ca_ca.csv': {'sha256': 'd44dcce79088a9aa0c2632b65ad9a6df55e54a784245f7698750b00002c41f76',
                                                                       'bytes': 457011781},
                 'Meta-IFD data/Ponzi_unbalance/edge/call_eoa_ca.csv': {'sha256': 'af18891948b53304ca68fa678731deb583a3783080d3ce632847a87e047cefc0',
                                                                        'bytes': 2397107086},
                 'Meta-IFD data/Ponzi_unbalance/edge/trans_ca_ca.csv': {'sha256': '65898127ae2286d9725b1f2c9f550ca0fb4ffaf30911b9c8263ef46271029d50',
                                                                        'bytes': 59202487},
                 'Meta-IFD data/Ponzi_unbalance/edge/trans_ca_eoa.csv': {'sha256': '0386b1967a814b7049176c028f1a80b590deff9fc227891b7194a1890e561992',
                                                                         'bytes': 241532233},
                 'Meta-IFD data/Ponzi_unbalance/edge/trans_eoa_ca.csv': {'sha256': '12c8465d0b4e84748f95e819fec10a3af56d055007eaabc58498581bf8590f49',
                                                                         'bytes': 325508430},
                 'Meta-IFD data/Ponzi_unbalance/edge/trans_eoa_eoa.csv': {'sha256': '1e8040229071cc700d13db38980910ec92840514078c89bc342bc70e26117f41',
                                                                          'bytes': 3215691170},
                 'Meta-IFD data/node_hete_EOA.csv': {'sha256': '0a0a2a601e81f8972622c4bdfdd81f070bebc57b65051303aa46a7d8ab52bcfb',
                                                     'bytes': 3422390643},
                 'Meta-IFD data/Phish_unbalance/EOA_labeled.csv': {'sha256': '5c3caee5fa88e083f208e8b8230069333e5133dd8376a727cdf611dcb3a80755',
                                                                   'bytes': 132781}},
 'attribute_provenance': None}

DECODER_REFERENCE=None


In [ ]:
DEFINITION_KEYS=['utilities.imports',
 'utilities.require',
 'utilities.sha256',
 'utilities.json_hash',
 'utilities.write_json',
 'utilities.read_json',
 'utilities.seed_everything',
 'utilities.runtime_information',
 'utilities.seal_files',
 'utilities.verify_files',
 'utilities.atomic_torch_save',
 'features.imports',
 'features.safe_ratio',
 'features.summarize_channel',
 'features.extract_sife44',
 'features.select_inputs',
 'data.imports',
 'data.HashedStream',
 'data.resolve_input',
 'data.open_raw_member',
 'data.obtain_graph',
 'data.verified_raw_cohort',
 'data.restore_float64_attributes',
 'data.prepare_data',
 'sampling.imports',
 'sampling.save_pyg_role_batches',
 'sampling.load_role_batches',
 'attention_conv.imports',
 'attention_conv.my_conv',
 'models_meta.imports',
 'models_meta.group',
 'models_meta.TripletLoss',
 'models_meta.HMSL',
 'views.imports',
 'views.FrozenInteractionDecoder',
 'views.decoder_tensor_fingerprint',
 'views.materialize_decoder',
 'views.author_row_normalize',
 'views.save_meta_views',
 'models.imports',
 'models.arm_columns',
 'models.transform_inputs',
 'models.StudyGAT',
 'models.StudySAGE',
 'models.build_classifier',
 'models.lookup_features',
 'models.forward_classifier',
 'metrics.imports',
 'metrics.classification_metrics',
 'tabular_inputs.imports',
 'tabular_inputs.tabular_column_names',
 'tabular_inputs.tabular_input_frame',
 'tabular_inputs.check_estimator_columns',
 'protocol.imports',
 'protocol.fixed_configuration',
 'protocol.supported_arms',
 'protocol.notebook_source_signature',
 'protocol.stratified_parts',
 'protocol.open_study',
 'protocol.open_split',
 'protocol.verify_split_lock',
 'protocol.verify_study_lock',
 'cache.imports',
 'cache.prepare_role',
 'cache.fit_training_statistics',
 'training.imports',
 'training.make_estimator',
 'training.tabular_predict',
 'training.neural_epoch',
 'training.fit_arm_seed',
 'training.SessionBudgetReached',
 'training.fit_split',
 'training.fit_all',
 'evaluation.imports',
 'evaluation.evaluate_trial',
 'evaluation.evaluate_all',
 'evaluation.summarize',
 'evaluation.export_results']


## 1. Verify inputs and compute exact strict-cohort features

Verify graph bytes before deserialization and raw label bytes. Remove all supervised rows belonging to duplicated addresses. Compute SIFE44 directly from full incident graph records,using the same reviewed definition on both datasets. Ponzi restores all six count/sum float64 arrays from raw,with independent byte hashes. Graph topology is retained for transductive sampling.

In [ ]:
SOURCE_SIGNATURE=notebook_source_signature()
RAW_SOURCE=resolve_input(DATA_RAW,DATASET,'raw')
PROCESSED_SOURCE=resolve_input(DATA_PHISH,DATASET,'processed')
print('Raw:',RAW_SOURCE,'| graph:',PROCESSED_SOURCE)
data=prepare_data(RAW_SOURCE,PROCESSED_SOURCE,DATA_OUTPUT,DATASET,REFERENCE,SOURCE_SIGNATURE)
print('Strict roots:',len(data['roots']['ids']),'| positives:',int(data['roots']['labels'].sum()),
      '| excluded IDs:',data['manifest']['excluded_ids'])


## 2. Freeze the fixed study contract

Every split/model seed,arm width,config,preprocessing,threshold,dependency version and scientific source cell is bound to this run. GAT uses the native signed-log/scaled backbone; GraphSAGE uses the Linear/tanh projected mean backbone. Source14 remains unchanged for SAGE/Meta/tree models. Pure36 has no source14. LightGBM uses the same named columns in fit and prediction. No grid or branch-specific winner selection.

In [ ]:
study=open_study(data,JOB_DIR,DATASET,MODEL,DEVICE,SOURCE_SIGNATURE,DECODER_REFERENCE,RUN_MODE,CPU_THREADS)
print('Configuration:',study['protocol']['config'])
print('Arms:',study['protocol']['arms'])
print('Unsupported arms:',study['protocol']['unsupported_arms'])
print('Expected fits:',len(study['protocol']['split_seeds'])*len(study['protocol']['model_seeds'])*len(study['protocol']['arms']))
print('Preprocessing:',study['protocol']['preprocessing'])


## 3. Fit TRAIN/validation on every partition and lock all arms

SIFE features are label-free and exact; GNN sampling only selects the message-passing topology. Within each split,all arms/seeds reuse the same sampled batches. Fit scalers only on TRAIN scope,choose each neural checkpoint using validation Macro-F1,and save last weights separately. Pure36 forwards no source14; Meta-IFD appends extras after its14dimensional decoder. Completed fits resume after hash verification.

In [ ]:
if notebook_source_signature()!=SOURCE_SIGNATURE:raise ValueError('Definitions changed during the run')
try:
    study_lock=fit_all(study,SESSION_DEADLINE)
except SessionBudgetReached:
    partial=OUTPUT_ROOT/(DATASET+'_'+MODEL+'_sife20_'+RUN_MODE+'_PARTIAL.zip')
    export_results(study,partial,allow_partial=True,include_caches=INCLUDE_LARGE_CACHES_IN_EXPORT)
    print('Saved partial progress:',partial,'| preserve full WORK_ROOT for exact resume:',WORK_ROOT)
    raise
print('All requested partitions/arms/seeds locked. TEST has not been scored by this fitting phase.')


## 4. Evaluate the frozen fits

Verify all partition locks before creating any TEST sampled batch. Restore saved estimators/checkpoints; never refit or select a threshold. Save per-address predictions and full per-seed metrics,then average model seeds within a split before summarizing20split means. Completed evaluation verifies saved artifacts instead of new inference.

In [ ]:
if notebook_source_signature()!=SOURCE_SIGNATURE:raise ValueError('Definitions changed after locking fits')
metrics=evaluate_all(study)
summary=pd.read_csv(JOB_DIR/'summary.csv')
display(summary[summary.metric=='macro_f1'])
display(pd.read_csv(JOB_DIR/'paired_macro_f1_deltas_pp.csv'))


## 5. Download results

The audit ZIP contains all fitted estimators/checkpoints/scalers,locks,memberships,predictions,metrics and split summaries. Large sampled topology/view caches are excluded by default; preserve WORK_ROOT or enable INCLUDE_LARGE_CACHES_IN_EXPORT for exact cached resume. Raw graph/large Ponzi sidecars are never duplicated into the download. Report all arms/splits,including decreases; overlapping partitions are not independent datasets.

In [ ]:
archive=OUTPUT_ROOT/(DATASET+'_'+MODEL+'_sife20_five_'+RUN_MODE+'_results.zip')
export_results(study,archive,include_caches=INCLUDE_LARGE_CACHES_IN_EXPORT)
print('COMPLETE:',RUN_MODE,'| download:',archive)
print('Keep for exact cached resume:',WORK_ROOT)
